# Laya V5.6 — Full then Delta: Two Optimized Five-Layer Liquid CeNN Models

[![Open in Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vtavakkoli/TinyCeNN-LM/blob/main/notebooks/Laya_V56_FiveLayer_FULL_Then_DELTA_LiquidCeNN_Colab.ipynb)

## Two DISTINCT models, trained and evaluated in order

1. **FULL Liquid CeNN** (runs first): at FFN output layers **20, 24, 23, 26, 8**, remove each original `mlp.Wo` projection and replace it with a CeNN-inspired liquid token-neighborhood cell. Pretrain each cell on actual FFN activations, installing them progressively; then jointly fine-tune all five on shuffled gold-labelled TRAIN with a teacher-KL regularizer. Each layer is a real independent replacement. Checkpoint and private HF target: `vtava/Laya-v56-FULL-5LiquidCeNN`.
2. **DELTA Liquid CeNN** (runs second): recreate the exact *untouched compressed baseline*, retain every original FFN projection, then attach five residual liquid cells using `Wo(x) + alpha*Liquid(x)`. All five alpha values begin at **zero** for mathematically exact initial parity; jointly train the five residuals with gold labels and KL. Independent checkpoint and private HF target: `vtava/Laya-v56-DELTA-5LiquidCeNN`.

Both variants retain the other ModernBERT attention and FFN components and the 28 encoder layers. **Neither is a full attention-to-CeNN transformation.** A residual DELTA CeNN has *more*, not fewer, parameters than the corresponding compressed `Wo` projection. Only FULL removes the five original `Wo` matrices.

## Why five layers rather than brute-forcing 28?

From the provided V5.5 results ZIP, the forced 28-FFN model fell to **31% gold on 200 test decisions**, with **0.845× GPU speed** vs compressed baseline. The greedily selected five-layer subset (20,24,23,26,8) obtained **77% test gold**, versus **75.5%** compressed baseline and **76.5%** teacher; its GPU speed was **0.974×**. The difference is only 2–3 decisions and **is not a reliable claim of improved generalization**. V5.5 supplied only statistics, not its learned five-cell weights, so V5.6 **reconstructs cells from scratch** using original frozen FFN activations rather than pretending to restore unavailable model weights.

## Experimental protocol

- One pinned source revision; one fixed TRAIN/PROBE/GOLD-VAL/TEST split; separate FULL and DELTA models trained on identical shuffled training data (same seed, independent optimizers).
- Each variant may select its best checkpoint on the 180-decision validation set; **no selection on TEST**, which is used only after freezing each model. Because earlier V5.x studies used the same TEST set, it is NOT a pristine cross-study holdout.
- Per-model release gates: **no gold-accuracy regression relative to compressed baseline**, sufficient teacher fidelity, bounded latency, strict standalone reload, and measured reduction relative to original teacher. FULL also requires checkpoint compression relative to compressed baseline; DELTA reports the parameter overhead and does not claim a compression gain from CeNN itself.
- The notebook runs **FULL first, then DELTA**, compares both side by side, and privately uploads each standalone model **only if that variant passes its own gates**. No model binary is sent to your computer.
- The last cell automatically downloads **`Laya_V56_FULL_vs_DELTA_Statistics.zip`**, including model selection, validation/test gold, paired speed, storage, quality gates and independent Hugging Face statuses—**no weights**.

All 28 ModernBERT encoder layers remain; only FFN output projections at the five selected layers change. This is a liquid/CeNN-inspired 1D token-lattice recurrence, not a 2D canonical Chua CeNN or a formally solved LTC ODE.

[Standalone source and strict two-variant loader](https://github.com/vtavakkoli/TinyCeNN-LM/blob/main/standalone/laya_v56_liquid_full_delta/modeling_laya_v56.py)


In [ ]:
#@title 1. Pinned requirements, GPU environment and runtime imports
%pip -q install "transformers==4.57.6" "datasets>=2.20" "huggingface_hub>=0.25,<1" "safetensors>=0.4"
import json,math,time,random,hashlib,copy,statistics,gc,importlib.util
from pathlib import Path
from collections import OrderedDict,Counter
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from datasets import load_dataset
from huggingface_hub import HfApi,snapshot_download,notebook_login
from safetensors.torch import save_file,load_file
from transformers import AutoTokenizer
assert torch.cuda.is_available(),"Select a Colab GPU (T4 or better)"
DEVICE=torch.device("cuda")
print("Torch:",torch.__version__,"GPU:",torch.cuda.get_device_name(0))


In [ ]:
#@title 2. FULL then DELTA experiment settings, independent HF model repositories
TEACHER_REPO="vtava/Laya-1.2-speedup-standalone"
PINNED_TEACHER_REVISION="b60184b2976e9f7ffa60c1c42b221d81b3691c3d"
FULL_HF_REPO="vtava/Laya-v56-FULL-5LiquidCeNN"
DELTA_HF_REPO="vtava/Laya-v56-DELTA-5LiquidCeNN"
SEED=2047
MAX_LEN=512
TRAIN_CASES=500
DEV_CASES=240
BATCH=2 #@param {type:"integer"}
TRAIN_B=2 #@param {type:"integer"}
TRAIN_MAX_TOKENS=224
GOLD_VAL_DECISIONS=180
HISTORICAL_PROBE_DECISIONS=120
AUDIT_DECISIONS=200
CAPTURE_BATCHES=5 #@param {type:"integer"}
LOCAL_DISTILL_STEPS=100 #@param {type:"integer"}
LOCAL_DISTILL_LR=0.001 #@param {type:"number"}
FULL_TRAIN_STEPS=220 #@param {type:"integer"}
DELTA_TRAIN_STEPS=220 #@param {type:"integer"}
TRAIN_CHECK_EVERY=20 #@param {type:"integer"}
FULL_LR=0.00008 #@param {type:"number"}
DELTA_LR=0.00020 #@param {type:"number"}
DELTA_SCALE_LR=0.002 #@param {type:"number"}
GOLD_CE_WEIGHT=0.75
GOLD_SOFT_WEIGHT=0.05
TEACHER_KL_WEIGHT=0.20
BENCH_REPEATS=10 #@param {type:"integer"}
MIN_TEACHER_AGREEMENT=0.85
MAX_TEST_JS=0.012
MIN_SPEED_RATIO=0.90
MIN_TOTAL_SAVED_BYTES=100_000_000
MIN_FULL_SAVED_VS_BASELINE_BYTES=1_000_000
BASELINE_V52_STANDALONE_DIR="" #@param {type:"string"}
PUBLISH_TO_HF=True #@param {type:"boolean"}
PRIVATE_HF_REPO=True #@param {type:"boolean"}
ALLOW_UPDATE_EXISTING=False #@param {type:"boolean"}
assert 1<=BATCH<=8 and 1<=TRAIN_B<=4
assert CAPTURE_BATCHES>=1 and LOCAL_DISTILL_STEPS>=1
assert TRAIN_CHECK_EVERY>=1 and FULL_TRAIN_STEPS>=0 and DELTA_TRAIN_STEPS>=0
assert abs(GOLD_CE_WEIGHT+GOLD_SOFT_WEIGHT+TEACHER_KL_WEIGHT-1)<1e-8
assert BENCH_REPEATS>=5
DEVICE=torch.device("cuda")
device=DEVICE
amp_dtype=(torch.bfloat16 if torch.cuda.is_bf16_supported(including_emulation=False)
           else torch.float16)
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
OUT=Path("/content/Laya_V56_FullThenDelta")
OUT.mkdir(exist_ok=True,parents=True)
api=HfApi()
try:
    teacher_info=api.model_info(TEACHER_REPO,revision=PINNED_TEACHER_REVISION)
except Exception:
    notebook_login()
    teacher_info=api.model_info(TEACHER_REPO,revision=PINNED_TEACHER_REVISION)
assert teacher_info.sha==PINNED_TEACHER_REVISION
parent_dir=Path(snapshot_download(TEACHER_REPO,revision=PINNED_TEACHER_REVISION))
source_config=json.loads((parent_dir/"config.json").read_text())
assert source_config["format"]=="laya-1p2-full-standalone-v1"
assert int(source_config["head_context_budget"])==64
source_cfg={"head_max_len":192,"max_len":MAX_LEN}
tokenizer=AutoTokenizer.from_pretrained(str(parent_dir/"tokenizer"),local_files_only=True)
SELECTED_LIQUID_SPECS=[
    dict(layer=20,width=64,steps=2,dt_bound=0.5),
    dict(layer=24,width=32,steps=2,dt_bound=0.5),
    dict(layer=23,width=16,steps=1,dt_bound=0.5),
    dict(layer=26,width=64,steps=2,dt_bound=0.5),
    dict(layer=8,width=16,steps=2,dt_bound=0.5)
]
assert len(SELECTED_LIQUID_SPECS)==5
assert len({c["layer"] for c in SELECTED_LIQUID_SPECS})==5
print("Five exact architectures from V5.5 greedy:",SELECTED_LIQUID_SPECS)
print("Original teacher:",teacher_info.sha)


In [ ]:
#@title 3. Embedded standalone V5.6 FULL/DELTA runtime and initial models
V56_RUNTIME_CODE = "\"\"\"Laya V5.0 all-28-layer brute-force hybrid SVD compression, fully standalone.\n\nRequires only torch, transformers, safetensors. Does NOT download or import\nconvaiinnovations/laya-typed-decisions or TinyCeNN-LM at inference.\nThe model weights and tokenizer must be present in the same export directory.\nOption-score model only; auxiliary action logits are intentionally unavailable.\n\"\"\"\nfrom __future__ import annotations\nimport json\nfrom pathlib import Path\nfrom typing import Any, Mapping, Sequence\nimport torch\nfrom torch import Tensor, nn\nfrom torch.nn import functional as F\nfrom safetensors.torch import load_file\nfrom transformers import AutoModel, AutoConfig, AutoTokenizer\n\nQTYPES = {\"choice\": 0, \"score\": 1, \"noul\": 2}\n_DEFAULT_NOUL_LABELS = {\"false\": \"false\", \"true\": \"true\"}\n\n\ndef _serialize_state(state: str | dict | list) -> str:\n    if isinstance(state, str):\n        return state\n    return json.dumps(state, ensure_ascii=False, separators=(\",\", \":\"), default=str)\n\n\ndef _render_value(value: Any) -> str:\n    if isinstance(value, str):\n        return value\n    return json.dumps(value, ensure_ascii=False, separators=(\", \", \": \"), default=str)\n\n\ndef normalize_question(question: Mapping[str, Any]) -> dict[str, Any]:\n    \"\"\"Normalize an external typed-decision question into the runtime format.\"\"\"\n    qtype = str(question.get(\"type\", question.get(\"t\", \"choice\"))).lower()\n    if qtype not in QTYPES:\n        raise ValueError(f\"unsupported question type: {qtype!r}\")\n\n    criteria = question.get(\"criteria\", question.get(\"crit\"))\n    if qtype == \"choice\":\n        if isinstance(criteria, (list, tuple)):\n            criteria = {str(x): None for x in criteria}\n        if not isinstance(criteria, Mapping) or len(criteria) < 2:\n            raise ValueError(\"choice questions require at least two criteria/options\")\n        criteria = dict(criteria)\n    elif qtype == \"score\":\n        if not isinstance(criteria, (list, tuple)) or len(criteria) < 2:\n            raise ValueError(\"score questions require at least two ordered criteria\")\n        criteria = list(criteria)\n    else:\n        if criteria is None:\n            criteria = {}\n        if not isinstance(criteria, Mapping):\n            raise ValueError(\"noul criteria must be a mapping when provided\")\n        criteria = {str(k).lower(): v for k, v in criteria.items()}\n\n    instructions = question.get(\"instructions\", question.get(\"ins\", \"\"))\n    if not isinstance(instructions, str):\n        instructions = json.dumps(instructions, ensure_ascii=False, default=str)\n\n    result = {\"t\": qtype, \"ins\": instructions, \"crit\": criteria}\n    if \"labels\" in question:\n        result[\"labels\"] = dict(question[\"labels\"])\n    return result\n\n\ndef render_options(question: Mapping[str, Any]) -> list[str]:\n    q = normalize_question(question) if \"t\" not in question else dict(question)\n    qtype, criteria = q[\"t\"], q.get(\"crit\")\n    if qtype == \"choice\":\n        return [\n            str(k) if v is None or v == \"\" else f\"{k}: {_render_value(v)}\"\n            for k, v in criteria.items()\n        ]\n    if qtype == \"score\":\n        return [f\"level {i}: {_render_value(v)}\" for i, v in enumerate(criteria)]\n\n    labels = q.get(\"labels\", _DEFAULT_NOUL_LABELS)\n    false_label = str(labels.get(\"false\", \"false\")).strip() or \"false\"\n    true_label = str(labels.get(\"true\", \"true\")).strip() or \"true\"\n    false_desc = criteria.get(\"false\")\n    true_desc = criteria.get(\"true\")\n    return [\n        f\"{false_label}: \" + (\n            _render_value(false_desc) if false_desc not in (None, \"\") else \"no, the statement does not hold\"\n        ),\n        f\"{true_label}: \" + (\n            _render_value(true_desc) if true_desc not in (None, \"\") else \"yes, the statement holds\"\n        ),\n    ]\n\n\ndef build_sequence(\n    tokenizer,\n    state: str | dict | list,\n    question: Mapping[str, Any],\n    *,\n    max_len: int = 512,\n    head_max_len: int = 192,\n    truncate_left: bool = False,\n    state_ids: Sequence[int] | None = None,\n) -> tuple[list[int], list[int]]:\n    \"\"\"Build a compact typed-decision sequence and option marker positions.\"\"\"\n    q = normalize_question(question) if \"t\" not in question else dict(question)\n    options = render_options(q)\n    mask_token = tokenizer.mask_token\n    instruction = str(q[\"ins\"]).replace(mask_token, \" \")\n    head_ids = tokenizer(\n        f\"{q['t']} question: {instruction}\", add_special_tokens=False\n    )[\"input_ids\"]\n\n    option_ids: list[list[int]] = []\n    for option in options:\n        ids = tokenizer(\n            \" \" + option.replace(mask_token, \" \"),\n            add_special_tokens=False,\n            truncation=True,\n            max_length=48,\n        )[\"input_ids\"]\n        option_ids.append([tokenizer.mask_token_id] + ids)\n\n    option_budget = head_max_len - sum(len(x) for x in option_ids)\n    if option_budget < 16:\n        per = max(4, (head_max_len - 16) // max(1, len(option_ids)))\n        option_ids = [x[:per] for x in option_ids]\n        option_budget = head_max_len - sum(len(x) for x in option_ids)\n    head_ids = head_ids[: max(8, option_budget)]\n\n    ids = [tokenizer.cls_token_id] + head_ids + [tokenizer.sep_token_id]\n    markers: list[int] = []\n    for option in option_ids:\n        markers.append(len(ids))\n        ids.extend(option)\n    ids.append(tokenizer.sep_token_id)\n\n    if state_ids is None:\n        text = _serialize_state(state).replace(mask_token, \" \")\n        state_ids = tokenizer(text, add_special_tokens=False)[\"input_ids\"]\n    room = max(0, max_len - len(ids) - 1)\n    if truncate_left:\n        start = max(0, len(state_ids) - room)\n        state_part = list(state_ids[start:])\n    else:\n        state_part = list(state_ids[:room])\n    ids = (ids + state_part + [tokenizer.sep_token_id])[:max_len]\n    markers = [m for m in markers if m < max_len]\n    return ids, markers\n\n\ndef collate_items(items: Sequence[Mapping[str, Any]], pad_id: int) -> dict[str, Tensor]:\n    if not items:\n        raise ValueError(\"items must not be empty\")\n    n = len(items)\n    length = max(len(x[\"ids\"]) for x in items)\n    kmax = max(len(x[\"markers\"]) for x in items)\n    ids = torch.full((n, length), int(pad_id), dtype=torch.long)\n    mask = torch.zeros((n, length), dtype=torch.long)\n    marker_pos = torch.zeros((n, kmax), dtype=torch.long)\n    marker_mask = torch.zeros((n, kmax), dtype=torch.bool)\n    qtype = torch.zeros(n, dtype=torch.long)\n\n    for i, item in enumerate(items):\n        row = torch.tensor(item[\"ids\"], dtype=torch.long)\n        ids[i, : row.numel()] = row\n        mask[i, : row.numel()] = 1\n        m = torch.tensor(item[\"markers\"], dtype=torch.long)\n        marker_pos[i, : m.numel()] = m\n        marker_mask[i, : m.numel()] = True\n        qtype[i] = int(item[\"qtype\"])\n    return {\n        \"input_ids\": ids,\n        \"attention_mask\": mask,\n        \"marker_pos\": marker_pos,\n        \"marker_mask\": marker_mask,\n        \"qtype\": qtype,\n    }\n\n\n\ndef _apply_rope_compat(cfg):\n    rope = getattr(cfg, \"rope_parameters\", None)\n    if not isinstance(rope,dict):\n        return\n    flat = rope.get(\"rope_theta\")\n    for layer_type, attr in ((\"full_attention\",\"global_rope_theta\"),\n                             (\"sliding_attention\",\"local_rope_theta\")):\n        v = rope.get(layer_type)\n        theta = v.get(\"rope_theta\") if isinstance(v,dict) else flat\n        if theta is not None and hasattr(cfg,attr):\n            setattr(cfg,attr,float(theta))\n\n\nclass LayaStandalone(nn.Module):\n    \"\"\"Native pretrained ModernBERT + exact 64-token option-scoring head.\n\n    ALL weights are stored locally in model.safetensors. No auxiliary act_head.\n    \"\"\"\n    def __init__(self, encoder_config=None, *, budget=64, head_layers=2,\n                 pretrained_components=None):\n        super().__init__()\n        self.budget = int(budget)\n        assert self.budget == 64, \"This checkpoint implements the validated 64-token head\"\n        if pretrained_components is not None:\n            # EXPORT ONLY: reuse the loaded upstream model without duplicating\n            # ~400M weights; this branch is not needed during standalone load.\n            self.encoder = pretrained_components.encoder\n            self.head = pretrained_components.head\n            self.type_emb = pretrained_components.type_emb\n            self.scorer = pretrained_components.scorer\n        else:\n            assert encoder_config is not None\n            _apply_rope_compat(encoder_config)\n            self.encoder = AutoModel.from_config(\n                encoder_config, attn_implementation=\"sdpa\")\n            d = int(encoder_config.hidden_size)\n            nhead = max(1,d//64)\n            layer = nn.TransformerEncoderLayer(\n                d,nhead,dim_feedforward=4*d,dropout=0.1,\n                batch_first=True,norm_first=True)\n            self.head = nn.TransformerEncoder(\n                layer,int(head_layers),enable_nested_tensor=False)\n            self.type_emb = nn.Embedding(3,d)\n            self.scorer = nn.Sequential(nn.LayerNorm(d),nn.Linear(d,d),\n                                        nn.GELU(),nn.Linear(d,1))\n\n    def _select_head_tokens(self,h,attention_mask,marker_pos,marker_mask):\n        # Copied from V4.6, keeping exact order/masks for reproduction.\n        b,t,d=h.shape\n        cls=torch.zeros((b,1),dtype=torch.long,device=h.device)\n        lm_count=min(self.budget,max(t-1,1))\n        lm=torch.linspace(1,max(t-1,1),steps=lm_count,device=h.device).round().long()\n        lm=lm.clamp(max=t-1).view(1,-1).expand(b,-1)\n        marker_ok=marker_mask.bool()\n        lm_ok=attention_mask.bool().gather(1,lm)\n        lm_ok=lm_ok & (lm!=0)\n        duplicate=(lm[:,:,None]==marker_pos[:,None,:]) & marker_ok[:,None,:]\n        lm_ok=lm_ok & ~duplicate.any(-1)\n        idx=torch.cat((cls,marker_pos.clamp(0,t-1),lm),dim=1)\n        keep=torch.cat((torch.ones((b,1),dtype=torch.bool,device=h.device),\n                        marker_ok,lm_ok),dim=1)\n        gathered=h.gather(1,idx[:,:,None].expand(-1,-1,d))\n        return gathered,keep\n\n    def forward(self,input_ids,attention_mask,marker_pos,marker_mask,qtype):\n        h=self.encoder(input_ids=input_ids,\n                       attention_mask=attention_mask).last_hidden_state\n        h=h+self.type_emb(qtype)[:,None,:]\n        compact,keep=self._select_head_tokens(\n            h,attention_mask,marker_pos,marker_mask)\n        transformed=self.head(compact,src_key_padding_mask=~keep)\n        marker_hidden=transformed[:,1:1+marker_pos.shape[1],:]\n        logits=self.scorer(marker_hidden).squeeze(-1).float()\n        logits=logits.masked_fill(~marker_mask,-1e4)\n        return logits,None\n\n    @torch.inference_mode()\n    def predict(self,tokenizer,state,questions,*,max_len=512,head_max_len=192):\n        items=[];names=[];labels=[]\n        for name,raw in questions.items():\n            q=normalize_question(raw)\n            if q[\"t\"]!=\"choice\":\n                raise ValueError(\"Only choice questions supported by predict()\")\n            opts=list(q[\"crit\"].keys())\n            ids,markers=build_sequence(tokenizer,state,q,max_len=max_len,\n                                       head_max_len=head_max_len)\n            if len(markers)!=len(opts):\n                raise ValueError(\"Truncated an option marker\")\n            names.append(name);labels.append(opts)\n            items.append(dict(ids=ids,markers=markers,qtype=QTYPES[q[\"t\"]]))\n        batch=collate_items(items,int(tokenizer.pad_token_id))\n        device=next(self.parameters()).device\n        batch={k:v.to(device) for k,v in batch.items()}\n        with (torch.autocast(\"cuda\",dtype=torch.float16)\n              if device.type==\"cuda\" else __import__(\"contextlib\").nullcontext()):\n            logits,_=self(**batch)\n        probs=logits.float().masked_fill(~batch[\"marker_mask\"],-1e9).softmax(-1).cpu()\n        answers={}\n        for i,name in enumerate(names):\n            p=probs[i,:len(labels[i])]\n            best=int(p.argmax())\n            answers[name]=dict(choice=labels[i][best],\n                               confidence=float(p[best]),\n                               probabilities={label:float(pr) for label,pr in zip(labels[i],p)})\n        return answers\n\n\ndef load_model(model_dir,*,device=\"cpu\",local_files_only=True,compile_head=False):\n    \"\"\"Load entirely from the local standalone directory. No network needed.\"\"\"\n    root=Path(model_dir).resolve()\n    metadata=json.loads((root/\"config.json\").read_text())\n    assert metadata[\"format\"]==\"laya-1p2-full-standalone-v1\"\n    enc_cfg=AutoConfig.from_pretrained(\n        str(root/\"encoder\"),local_files_only=local_files_only)\n    model=LayaStandalone(enc_cfg,budget=int(metadata[\"head_context_budget\"]),\n                         head_layers=int(metadata[\"head_layers\"]))\n    state=load_file(str(root/\"model.safetensors\"),device=\"cpu\")\n    model.load_state_dict(state,strict=True)\n    model=model.to(device).eval().requires_grad_(False)\n    model.encoder.config.reference_compile=False\n    if compile_head:\n        model.head=torch.compile(model.head,mode=\"reduce-overhead\",dynamic=False)\n    tok=AutoTokenizer.from_pretrained(\n        str(root/\"tokenizer\"),local_files_only=local_files_only)\n    return model,tok\n\ndef local_pair_indices(model):\n    \"\"\"Verify FULL / same-window LOCAL / LOCAL triples in ModernBERT.\n\n    transformers==4.57.6 uses attn.local_attention: (-1,-1) means FULL,\n    (64,64) is a 128-token sliding window. Newer builds can expose\n    config.layer_types with attn.sliding_window instead. Support both.\n    \"\"\"\n    layers=model.encoder.layers\n    cfg=model.encoder.config\n    n=len(layers)\n    layer_types=getattr(cfg,\"layer_types\",None)\n    if layer_types is not None and len(layer_types)!=n:\n        raise ValueError(\"ModernBERT layer_types count does not match layers\")\n\n    def mode_at(i):\n        a=layers[i].attn\n        old_window=getattr(a,\"local_attention\",None)\n        if old_window is not None:\n            window=tuple(old_window)\n            if window==(-1,-1):\n                return (\"full\",None)\n            if len(window)!=2 or min(window)<0:\n                raise ValueError(f\"Malformed local attention window at {i}: {window}\")\n            return (\"local\",window)\n        typ=(layer_types[i] if layer_types is not None else\n             getattr(layers[i],\"attention_type\",None))\n        window=getattr(a,\"sliding_window\",None)\n        if typ==\"full_attention\" and window is None:\n            return (\"full\",None)\n        if typ==\"sliding_attention\" and window is not None:\n            return (\"local\",int(window))\n        raise ValueError(f\"Unsupported ModernBERT attention metadata at layer {i}: {typ}, {window}\")\n\n    pairs=[]\n    for anchor in range(0,n,3):\n        mode,_=mode_at(anchor)\n        if mode!=\"full\":\n            raise ValueError(f\"Layer {anchor} must be FULL attention\")\n        left_idx,right_idx=anchor+1,anchor+2\n        if right_idx>=n:\n            continue\n        m1,w1=mode_at(left_idx)\n        m2,w2=mode_at(right_idx)\n        if m1!=\"local\" or m2!=\"local\" or w1!=w2:\n            raise ValueError(\n                f\"Layers {(left_idx,right_idx)} must have identical local windows: {m1}/{w1}, {m2}/{w2}\")\n        pairs.append((left_idx,right_idx))\n    return pairs\n\n\nclass SVDDeltaLinear(nn.Module):\n    \"\"\"Second-layer linear projection = shared first-layer projection + A @ B.\n\n    The shared linear is a plain Python reference, NOT a registered submodule;\n    its weights appear only once in state_dict, under the original first layer.\n    \"\"\"\n    def __init__(self, first_linear, rank, *, A=None, B=None):\n        super().__init__()\n        if not isinstance(first_linear,nn.Linear):\n            raise TypeError(f\"Expected nn.Linear, got {type(first_linear)}\")\n        out_dim,in_dim=first_linear.weight.shape\n        rank=int(rank)\n        if not (1<=rank<min(in_dim,out_dim)):\n            raise ValueError(f\"Unsupported rank {rank} for {out_dim}x{in_dim}\")\n        # Using object.__setattr__ prevents double registering the shared module.\n        object.__setattr__(self,\"_shared\",first_linear)\n        self.in_features=int(in_dim)\n        self.out_features=int(out_dim)\n        self.max_rank=rank\n        self.active_rank=rank\n        self.A=nn.Parameter(torch.zeros((out_dim,rank),device=first_linear.weight.device,\n                                        dtype=first_linear.weight.dtype))\n        self.B=nn.Parameter(torch.zeros((rank,in_dim),device=first_linear.weight.device,\n                                        dtype=first_linear.weight.dtype))\n        if A is not None and B is not None:\n            with torch.no_grad():\n                self.A.copy_(A.to(self.A.device,self.A.dtype))\n                self.B.copy_(B.to(self.B.device,self.B.dtype))\n\n    def set_rank(self, rank):\n        rank=int(rank)\n        if rank<1 or rank>self.max_rank:\n            raise ValueError(f\"rank {rank} outside 1..{self.max_rank}\")\n        self.active_rank=rank\n\n    def shrink(self, rank):\n        \"\"\"Permanently discard extra SVD columns for a compact checkpoint.\"\"\"\n        rank=int(rank)\n        if rank<1 or rank>self.max_rank:\n            raise ValueError(rank)\n        self.A=nn.Parameter(self.A[:,:rank].detach().clone())\n        self.B=nn.Parameter(self.B[:rank,:].detach().clone())\n        self.max_rank=rank\n        self.active_rank=rank\n\n    def forward(self,x):\n        # fp16 autocast handles both paths identically, with FP32 stored weights.\n        w=self._shared.weight\n        bias=self._shared.bias\n        base=F.linear(x,w,bias)\n        r=self.active_rank\n        return base+F.linear(F.linear(x,self.B[:r,:]),self.A[:,:r])\n\n\n\n# V4.9 does NOT force compression of all nine local pairs.\n# Each selected follower projection gets a separate low-rank residual;\n# all unselected attention projection matrices retain pretrained weights.\ndef projection_slots(model):\n    slots=[]\n    for first,second in local_pair_indices(model):\n        for component in (\"Wqkv\",\"Wo\"):\n            a=getattr(model.encoder.layers[first].attn,component)\n            b=getattr(model.encoder.layers[second].attn,component)\n            if not isinstance(a,nn.Linear) or not isinstance(b,nn.Linear):\n                raise TypeError(f\"Expected nn.Linear for layer {second}.{component}\")\n            if a.weight.shape != b.weight.shape:\n                raise ValueError(f\"Incompatible original projection: {second}.{component}\")\n            if a.bias is not None or b.bias is not None:\n                raise ValueError(f\"Biasful projection needs separate residual bias: {second}.{component}\")\n            slots.append((first,second,component))\n    return slots\n\n\ndef _validate_slot(model, first, second, component):\n    \"\"\"Validate schedule independently of other already-compressed slots.\"\"\"\n    first,second=int(first),int(second)\n    component=str(component)\n    if (first,second) not in local_pair_indices(model):\n        raise ValueError(f\"Unknown local pair: {(first,second)}\")\n    if component not in (\"Wqkv\",\"Wo\"):\n        raise ValueError(f\"Unknown projection: {component}\")\n    return first,second,component\n\n\ndef svd_factors(model, first, second, component, max_rank=256,\n                oversampling=8, niter=2):\n    \"\"\"Compute one factorization per projection from ORIGINAL pretrained weights.\n\n    A and B are float32 CPU tensors and can be sliced for different ranks.\n    \"\"\"\n    first,second,component=_validate_slot(model,first,second,component)\n    a=getattr(model.encoder.layers[first].attn,component)\n    b=getattr(model.encoder.layers[second].attn,component)\n    if not isinstance(a,nn.Linear) or not isinstance(b,nn.Linear):\n        raise ValueError(\"SVD must be done BEFORE that follower is compressed\")\n    with torch.no_grad():\n        delta=b.weight.detach().float()-a.weight.detach().float()\n        r=min(int(max_rank),min(delta.shape)-1)\n        q=min(r+int(oversampling),min(delta.shape))\n        # Randomized SVD (approximation, not an exact mathematical SVD).\n        U,S,V=torch.svd_lowrank(delta,q=q,niter=int(niter))\n        root=S[:r].clamp_min(0).sqrt()\n        A=(U[:,:r]*root.unsqueeze(0)).cpu().contiguous()\n        B=(root.unsqueeze(1)*V[:,:r].T).cpu().contiguous()\n        total=float(delta.square().sum().item())\n        error=float((delta-(A.to(delta.device)@B.to(delta.device))).square().sum().item())\n        del U,S,V,delta\n    return A,B,{\"rank\":r,\n        \"relative_frobenius_error\":(error/max(total,1e-30))**0.5,\n        \"saved_parameter_bytes_if_rank256\":None}\n\n\ndef build_adapter(model, first, second, component, rank, factors=None,\n                  *, initialized=True):\n    \"\"\"Replace ONE follower's projection; return original module for rollback.\n\n    If initialized=False, A/B are zeros until strictly loaded from checkpoint.\n    \"\"\"\n    first=int(first);second=int(second);component=str(component)\n    if (first,second) not in local_pair_indices(model):\n        raise ValueError(f\"Invalid local pair: {(first,second)}\")\n    leader=getattr(model.encoder.layers[first].attn,component)\n    follower_block=model.encoder.layers[second].attn\n    old=getattr(follower_block,component)\n    if not isinstance(leader,nn.Linear) or not isinstance(old,nn.Linear):\n        raise TypeError(\"Projection already wrapped or unavailable\")\n    if leader.bias is not None or old.bias is not None:\n        raise ValueError(\"Biasful attention projections are unsupported\")\n    if initialized:\n        if factors is None:\n            A,B,_=svd_factors(model,first,second,component,max_rank=rank)\n        else:\n            A,B=factors\n        adapter=SVDDeltaLinear(leader,rank,A=A[:,:rank],B=B[:rank,:])\n    else:\n        adapter=SVDDeltaLinear(leader,rank)\n    setattr(follower_block,component,adapter)\n    return old\n\n\ndef restore_original_projection(model, second, component, original):\n    if not isinstance(original,nn.Linear):\n        raise TypeError(\"Rollback needs the original pretrained linear module\")\n    setattr(model.encoder.layers[int(second)].attn,str(component),original)\n\n\ndef selected_adapters(model):\n    selected=[]\n    for first,second in local_pair_indices(model):\n        for component in (\"Wqkv\",\"Wo\"):\n            layer=getattr(model.encoder.layers[second].attn,component)\n            if isinstance(layer,SVDDeltaLinear):\n                selected.append((first,second,component,layer))\n    return selected\n\n\ndef unique_parameter_count(model):\n    return sum(p.numel() for p in model.parameters())\n\n\ndef estimated_saved_bytes(model, first, second, component, rank):\n    leader=getattr(model.encoder.layers[first].attn,component)\n    outdim,indim=leader.weight.shape\n    # Full follower weight removed; A (out×rank), B (rank×in) remain.\n    return 4*(outdim*indim-int(rank)*(outdim+indim))\n\n\ndef load_v49_model(model_dir, *, device=\"cpu\", compile_head=False):\n    \"\"\"Strict, fully offline restoration of selectively compressed model.\"\"\"\n    root=Path(model_dir).resolve()\n    meta=json.loads((root/\"config.json\").read_text(encoding=\"utf-8\"))\n    if meta[\"format\"]!=\"laya-v49-selective-svd-v1\":\n        raise ValueError(\"Wrong model format: expected V4.9 selective SVD\")\n    cfg=AutoConfig.from_pretrained(str(root/\"encoder\"),local_files_only=True)\n    model=LayaStandalone(cfg,budget=int(meta[\"head_context_budget\"]),\n                         head_layers=int(meta[\"head_layers\"]))\n    for item in meta[\"compressed_projections\"]:\n        build_adapter(model,int(item[\"first\"]),int(item[\"second\"]),\n                      str(item[\"component\"]),int(item[\"rank\"]),\n                      initialized=False)\n    weights=load_file(str(root/\"model.safetensors\"),device=\"cpu\")\n    model.load_state_dict(weights,strict=True)\n    # No duplicated original follower weights or duplicated leader submodule.\n    assert len(selected_adapters(model))==len(meta[\"compressed_projections\"])\n    for first,second,component,wrapper in selected_adapters(model):\n        leader=getattr(model.encoder.layers[first].attn,component)\n        assert wrapper._shared is leader\n        assert \"_shared\" not in wrapper._modules\n    model=model.to(device).eval().requires_grad_(False)\n    model.encoder.config.reference_compile=False\n    if compile_head:\n        model.head=torch.compile(model.head,mode=\"reduce-overhead\",dynamic=False)\n    tokenizer=AutoTokenizer.from_pretrained(str(root/\"tokenizer\"),local_files_only=True)\n    return model,tokenizer\n\n\n# ---------------------------------------------------------------------------\n# V5.0: brute-force all 28 ModernBERT blocks, attention + FFN, preserving the\n# proven V4.9 pretrained-pair DELTA path as one selectable candidate family.\n# ---------------------------------------------------------------------------\n\nclass IndependentSVDLinear(nn.Module):\n    \"\"\"Plain original projection compressed as A @ B; original dense W omitted.\n\n    Compatible with ModernBERT's attention Wqkv, Wo and MLP Wi, Wo calls.\n    A saved bias is kept if the original linear was biased.\n    \"\"\"\n    def __init__(self, original: nn.Linear, rank: int, A=None, B=None):\n        super().__init__()\n        if not isinstance(original, nn.Linear):\n            raise TypeError(\"Expected original nn.Linear\")\n        out_dim,in_dim=original.weight.shape\n        rank=int(rank)\n        if rank<=0 or rank>=min(out_dim,in_dim):\n            raise ValueError(f\"Bad SVD rank {rank} for {out_dim}x{in_dim}\")\n        self.in_features=int(in_dim)\n        self.out_features=int(out_dim)\n        self.rank=rank\n        self.A=nn.Parameter(torch.zeros((out_dim,rank),\n            dtype=original.weight.dtype,device=original.weight.device))\n        self.B=nn.Parameter(torch.zeros((rank,in_dim),\n            dtype=original.weight.dtype,device=original.weight.device))\n        if original.bias is None:\n            self.register_parameter(\"bias\",None)\n        else:\n            self.bias=nn.Parameter(original.bias.detach().clone())\n        if A is not None and B is not None:\n            with torch.no_grad():\n                self.A.copy_(A[:,:rank].to(device=self.A.device,dtype=self.A.dtype))\n                self.B.copy_(B[:rank,:].to(device=self.B.device,dtype=self.B.dtype))\n\n    def forward(self,x):\n        return F.linear(F.linear(x,self.B),self.A,self.bias)\n\n\ndef all_encoder_slots(model):\n    \"\"\"Enumerate available projections in EVERY pretrained encoder layer.\"\"\"\n    slots=[]\n    for index,layer in enumerate(model.encoder.layers):\n        for block,names in ((\"attn\",(\"Wqkv\",\"Wo\")),(\"mlp\",(\"Wi\",\"Wo\"))):\n            component=getattr(layer,block)\n            for name in names:\n                module=getattr(component,name,None)\n                if module is None:\n                    raise ValueError(f\"Required projection layer {index}.{block}.{name} missing\")\n                if not isinstance(module,nn.Linear):\n                    raise TypeError(f\"Not an uncompressed nn.Linear: {index}.{block}.{name}: {type(module)}\")\n                slots.append((index,block,name))\n    if len(slots)!=4*len(model.encoder.layers):\n        raise RuntimeError(\"Expected exactly two attention and two FFN projections per layer\")\n    return slots\n\n\ndef list_candidate_families(model):\n    \"\"\"Full brute-force search space: 112 own-SVD + 18 delta-shared slots.\"\"\"\n    options=[{\"layer\":int(layer),\"block\":block,\"component\":name,\n              \"mode\":\"own_svd\"}\n             for layer,block,name in all_encoder_slots(model)]\n    for first,second in local_pair_indices(model):\n        for component in (\"Wqkv\",\"Wo\"):\n            options.append({\"layer\":int(second),\"leader\":int(first),\n                            \"block\":\"attn\",\"component\":component,\n                            \"mode\":\"pair_delta\"})\n    return options\n\n\ndef own_svd_factors(original, *, max_rank=256, oversampling=8,niter=2):\n    \"\"\"Randomized, truncated SVD of ORIGINAL pretrained W (not layer delta).\"\"\"\n    if not isinstance(original,nn.Linear):\n        raise TypeError(\"Expected an uncompressed linear projection\")\n    with torch.no_grad():\n        weight=original.weight.detach().float()\n        rank=min(int(max_rank),min(weight.shape)-1)\n        q=min(rank+int(oversampling),min(weight.shape))\n        U,S,V=torch.svd_lowrank(weight,q=q,niter=int(niter))\n        scale=S[:rank].clamp_min(0).sqrt()\n        A=(U[:,:rank]*scale[None,:]).cpu().contiguous()\n        B=(scale[:,None]*V[:,:rank].T).cpu().contiguous()\n        approximated=A.to(weight.device)@B.to(weight.device)\n        numerator=float((weight-approximated).square().sum().item())\n        denominator=float(weight.square().sum().item())\n    return A,B,{\"rank\":rank,\n                \"relative_frobenius_error\":(numerator/max(denominator,1e-30))**.5}\n\n\ndef projection_at(model, spec):\n    return getattr(getattr(model.encoder.layers[int(spec[\"layer\"])],\n                           str(spec[\"block\"])),str(spec[\"component\"]))\n\n\ndef replace_projection(model, spec, *, factors=None, initialized=True):\n    \"\"\"Apply one projection replacement; return original module for rollback.\"\"\"\n    idx=int(spec[\"layer\"])\n    block=str(spec[\"block\"])\n    comp=str(spec[\"component\"])\n    rank=int(spec[\"rank\"])\n    mode=str(spec[\"mode\"])\n    owner=getattr(model.encoder.layers[idx],block)\n    old=getattr(owner,comp)\n    if not isinstance(old,nn.Linear):\n        raise TypeError(f\"Already compressed or unsupported {idx}.{block}.{comp}\")\n    if mode==\"own_svd\":\n        if initialized and factors is None:\n            factors=own_svd_factors(old,max_rank=rank)[:2]\n        if initialized:\n            A,B=factors\n            replacement=IndependentSVDLinear(old,rank,A,B)\n        else:\n            replacement=IndependentSVDLinear(old,rank)\n    elif mode==\"pair_delta\":\n        if block!=\"attn\" or comp not in (\"Wqkv\",\"Wo\"):\n            raise ValueError(\"Pair delta only supported for local attention\")\n        leader=int(spec[\"leader\"])\n        if (leader,idx) not in local_pair_indices(model):\n            raise ValueError(f\"Not a valid local pair: {leader},{idx}\")\n        shared=getattr(model.encoder.layers[leader].attn,comp)\n        if not isinstance(shared,nn.Linear):\n            raise TypeError(\"Leader must retain original pretrained linear\")\n        if shared.bias is not None or old.bias is not None:\n            raise ValueError(\"Pair delta only supported for bias-free attention\")\n        if initialized:\n            if factors is None:\n                A,B,_=svd_factors(model,leader,idx,comp,max_rank=rank)\n            else:\n                A,B=factors\n            replacement=SVDDeltaLinear(shared,rank,A=A[:,:rank],B=B[:rank,:])\n        else:\n            replacement=SVDDeltaLinear(shared,rank)\n    else:\n        raise ValueError(f\"Unsupported family: {mode}\")\n    setattr(owner,comp,replacement)\n    return old\n\n\ndef revert_projection(model,spec,original):\n    if not isinstance(original,nn.Linear):\n        raise TypeError(\"Original projection must be nn.Linear\")\n    owner=getattr(model.encoder.layers[int(spec[\"layer\"])],str(spec[\"block\"]))\n    setattr(owner,str(spec[\"component\"]),original)\n\n\ndef applied_adapters(model,selected_specs):\n    result=[]\n    for spec in selected_specs:\n        wrapper=projection_at(model,spec)\n        if not isinstance(wrapper,(IndependentSVDLinear,SVDDeltaLinear)):\n            raise AssertionError(f\"Missing compressed adapter for {spec}\")\n        result.append(wrapper)\n    return result\n\n\ndef expected_saved_bytes(model,spec):\n    \"\"\"FP32 parameter saving for one replacement, including any old bias.\"\"\"\n    idx=int(spec[\"layer\"])\n    owner=getattr(model.encoder.layers[idx],str(spec[\"block\"]))\n    original=getattr(owner,str(spec[\"component\"]))\n    if not isinstance(original,nn.Linear):\n        raise TypeError(\"Call before installing candidate\")\n    out_dim,in_dim=original.weight.shape\n    rank=int(spec[\"rank\"])\n    # Bias exists in both old and new for own_svd; cancels out.\n    # Pair delta has no bias.\n    return int(4*(out_dim*in_dim-rank*(out_dim+in_dim)))\n\n\ndef load_v50_model(model_dir, *, device=\"cpu\",compile_head=False):\n    \"\"\"Strict reload of all hybrid low-rank projections from local disk only.\"\"\"\n    root=Path(model_dir).resolve()\n    info=json.loads((root/\"config.json\").read_text(encoding=\"utf-8\"))\n    if info[\"format\"]!=\"laya-v50-all28-bruteforce-v1\":\n        raise ValueError(\"Not a V5.0 hybrid model\")\n    cfg=AutoConfig.from_pretrained(str(root/\"encoder\"),local_files_only=True)\n    model=LayaStandalone(cfg,budget=int(info[\"head_context_budget\"]),\n                         head_layers=int(info[\"head_layers\"]))\n    # Rebuild in the same order: only uncompressed leaders are valid for delta.\n    for spec in info[\"compressed_projections\"]:\n        replace_projection(model,spec,initialized=False)\n    full=load_file(str(root/\"model.safetensors\"),device=\"cpu\")\n    model.load_state_dict(full,strict=True)\n    for spec,adapter in zip(info[\"compressed_projections\"],\n                            applied_adapters(model,info[\"compressed_projections\"])):\n        if spec[\"mode\"]==\"pair_delta\":\n            leader=getattr(model.encoder.layers[int(spec[\"leader\"])].attn,\n                           str(spec[\"component\"]))\n            assert adapter._shared is leader\n            assert \"_shared\" not in adapter._modules\n    model.to(device).eval().requires_grad_(False)\n    model.encoder.config.reference_compile=False\n    if compile_head:\n        model.head=torch.compile(model.head,mode=\"reduce-overhead\",dynamic=False)\n    tok=AutoTokenizer.from_pretrained(str(root/\"tokenizer\"),local_files_only=True)\n    return model,tok\n\n\n# ================================================================\n# Laya V5.3: CeNN-inspired token-neighborhood liquid recurrence\n# ================================================================\n\nclass LiquidCeNNProjection(nn.Module):\n    \"\"\"Replace/augment an FFN output projection with a 1D liquid CeNN cell.\n\n    This is CeNN-*inspired*: local nearest-neighbor coupling via depthwise\n    Conv1d over tokens plus gated state relaxation. It is not the original\n    continuous-time Chua CeNN equation, nor an LTC ODE solver.\n\n    Modes:\n      * replace: remove the old projection entirely, f(x)=liquid(x).\n      * residual: keep the original projection, f(x)=base(x)+scale*liquid(x).\n\n    The residual scale is initially zero, so this mode is exactly the\n    pretrained baseline at creation time, including identical predictions.\n    \"\"\"\n    def __init__(self,original,width=64,steps=3,mode=\"residual\",\n                 dt_bound=0.50):\n        super().__init__()\n        if not hasattr(original,\"in_features\") or not hasattr(original,\"out_features\"):\n            raise TypeError(\"LiquidCeNN requires a linear-like FFN projection\")\n        self.in_features=int(original.in_features)\n        self.out_features=int(original.out_features)\n        self.width=int(width)\n        self.steps=int(steps)\n        self.mode=str(mode)\n        self.dt_bound=float(dt_bound)\n        if self.width<4 or self.steps<1 or self.steps>10:\n            raise ValueError(\"Bad cell width or recurrent step count\")\n        if self.mode not in (\"replace\",\"residual\"):\n            raise ValueError(self.mode)\n        if not (0.0<self.dt_bound<=1.0):\n            raise ValueError(\"dt_bound must be in (0,1]\")\n        # Only register the original dense/low-rank projection in residual\n        # mode; replacement genuinely deletes its weights from state_dict.\n        self.base=original if self.mode==\"residual\" else None\n        self.drive=nn.Linear(self.in_features,self.width,bias=True)\n        self.drive_norm=nn.LayerNorm(self.width)\n        self.neighbor=nn.Conv1d(self.width,self.width,kernel_size=3,\n                                padding=1,groups=self.width,bias=False)\n        self.recurrent=nn.Linear(self.width,self.width,bias=False)\n        self.liquid_gate=nn.Linear(2*self.width,self.width)\n        self.readout=nn.Linear(self.width,self.out_features,bias=True)\n        # Start with small dynamic corrections instead of random large logits.\n        nn.init.normal_(self.readout.weight,mean=0.,std=0.008)\n        nn.init.zeros_(self.readout.bias)\n        self.scale=nn.Parameter(torch.tensor(0.,dtype=torch.float32))\n\n        # Match original device and dtype without moving the original base.\n        original_parameter=next(original.parameters(),None)\n        if original_parameter is not None:\n            for module in [self.drive,self.drive_norm,self.neighbor,\n                           self.recurrent,self.liquid_gate,self.readout]:\n                module.to(device=original_parameter.device,dtype=original_parameter.dtype)\n\n    def liquid(self,x):\n        # x must have [batch, tokens, d_ffn]. Spatial nearest neighbors\n        # are token neighbors, not a two-dimensional image lattice.\n        if x.ndim!=3:\n            raise ValueError(f\"Expected [B,T,D] FFN activation, got {tuple(x.shape)}\")\n        u=torch.tanh(self.drive_norm(self.drive(x)))\n        h=torch.zeros_like(u)\n        for _ in range(self.steps):\n            neighbors=self.neighbor(h.transpose(1,2)).transpose(1,2)\n            update_gate=torch.sigmoid(self.liquid_gate(torch.cat((u,h),dim=-1)))\n            dt=self.dt_bound*update_gate\n            proposal=torch.tanh(u+neighbors+self.recurrent(h))\n            h=h+dt*(proposal-h)\n        return self.readout(h)\n\n    def forward(self,x):\n        correction=self.liquid(x)\n        if self.mode==\"replace\":\n            return correction\n        # Alpha=0 is an EXACT base-model control despite random CeNN weights.\n        return self.base(x)+self.scale.to(dtype=correction.dtype)*correction\n\n\ndef install_liquid_ffn(model,layer_index,*,width=64,steps=3,mode=\"residual\",\n                       dt_bound=0.5):\n    layer_index=int(layer_index)\n    if not (0<=layer_index<len(model.encoder.layers)):\n        raise IndexError(layer_index)\n    owner=model.encoder.layers[layer_index].mlp\n    previous=owner.Wo\n    if isinstance(previous,LiquidCeNNProjection):\n        raise ValueError(f\"Liquid layer {layer_index} already installed\")\n    candidate=LiquidCeNNProjection(previous,width,steps,mode,dt_bound)\n    owner.Wo=candidate\n    return previous\n\n\ndef uninstall_liquid_ffn(model,layer_index,previous):\n    layer_index=int(layer_index)\n    if not isinstance(previous,nn.Module):\n        raise TypeError(\"Replacement rollback must restore nn.Module\")\n    module=model.encoder.layers[layer_index].mlp\n    if not isinstance(module.Wo,LiquidCeNNProjection):\n        raise ValueError(\"Current layer is not LiquidCeNNProjection\")\n    module.Wo=previous\n\n\ndef find_liquid_cells(model):\n    return [(i,block.mlp.Wo)\n            for i,block in enumerate(model.encoder.layers)\n            if isinstance(block.mlp.Wo,LiquidCeNNProjection)]\n\n\ndef load_v53_model(model_dir, *,device=\"cpu\",compile_head=False):\n    \"\"\"Fully offline strict reconstruction of compressed V5.3 + liquid CeNN.\"\"\"\n    root=Path(model_dir).resolve()\n    metadata=json.loads((root/\"config.json\").read_text(encoding=\"utf-8\"))\n    if metadata.get(\"format\")!=\"laya-v53-liquid-cenn-v1\":\n        raise ValueError(\"Not a V5.3 liquid CeNN export\")\n    cfg=AutoConfig.from_pretrained(str(root/\"encoder\"),local_files_only=True)\n    model=LayaStandalone(cfg,budget=int(metadata[\"head_context_budget\"]),\n                         head_layers=int(metadata[\"head_layers\"]))\n    for spec in metadata[\"compressed_projections\"]:\n        replace_projection(model,spec,initialized=False)\n    spec=metadata[\"liquid_cell\"]\n    install_liquid_ffn(model,int(spec[\"layer\"]),width=int(spec[\"width\"]),\n       steps=int(spec[\"steps\"]),mode=str(spec[\"mode\"]),\n       dt_bound=float(spec[\"dt_bound\"]))\n    state=load_file(str(root/\"model.safetensors\"),device=\"cpu\")\n    model.load_state_dict(state,strict=True)\n    assert len(find_liquid_cells(model))==1\n    model=model.to(device).eval().requires_grad_(False)\n    model.encoder.config.reference_compile=False\n    if compile_head:\n        model.head=torch.compile(model.head,mode=\"reduce-overhead\",dynamic=False)\n    tokenizer=AutoTokenizer.from_pretrained(\n       str(root/\"tokenizer\"),local_files_only=True)\n    return model,tokenizer\n\n\n# ================================================================\n# V5.5 all-28 FFN-output CeNN substitution (no attention substitution)\n# ================================================================\ndef load_v55_model(model_dir, *,device=\"cpu\",compile_head=False):\n    \"\"\"Load fully standalone model with a subset or all 28 FFN liquid cells.\n\n    Each selected encoder layer substitutes its mlp.Wo projection. Attention,\n    residual connections, norms, and all other FFNs remain ModernBERT.\n    \"\"\"\n    root=Path(model_dir).resolve()\n    info=json.loads((root/\"config.json\").read_text(encoding=\"utf-8\"))\n    if info.get(\"format\")!=\"laya-v55-all28-liquid-cenn-v1\":\n        raise ValueError(\"This checkpoint is not an all-layer CeNN V5.5 export\")\n    cfg=AutoConfig.from_pretrained(str(root/\"encoder\"),local_files_only=True)\n    model=LayaStandalone(cfg,budget=int(info[\"head_context_budget\"]),\n                         head_layers=int(info[\"head_layers\"]))\n    for spec in info[\"compressed_projections\"]:\n        replace_projection(model,spec,initialized=False)\n    liquid_specs=info[\"liquid_cells\"]\n    assert len(liquid_specs)==len({int(s[\"layer\"]) for s in liquid_specs})\n    assert all(s.get(\"mode\")==\"replace\" for s in liquid_specs)\n    for cell in liquid_specs:\n        install_liquid_ffn(model,int(cell[\"layer\"]),\n                          width=int(cell[\"width\"]),\n                          steps=int(cell[\"steps\"]),\n                          mode=\"replace\",dt_bound=float(cell[\"dt_bound\"]))\n    full=load_file(str(root/\"model.safetensors\"),device=\"cpu\")\n    model.load_state_dict(full,strict=True)\n    assert len(find_liquid_cells(model))==len(liquid_specs)\n    model.to(device).eval().requires_grad_(False)\n    model.encoder.config.reference_compile=False\n    if compile_head:\n        model.head=torch.compile(model.head,mode=\"reduce-overhead\",dynamic=False)\n    tokenizer=AutoTokenizer.from_pretrained(\n        str(root/\"tokenizer\"),local_files_only=True)\n    return model,tokenizer\n\n\n# ================================================================\n# V5.6 dual-model loader: FULL projection substitution vs DELTA addition\n# ================================================================\ndef load_v56_model(model_dir, *,device=\"cpu\",compile_head=False):\n    \"\"\"Strict, offline load of ONE of the five-layer liquid-CeNN variants.\n\n    FULL: 5 FFN Wo modules are entirely replaced by CeNN recurrence.\n    DELTA: 5 compressed original FFN Wo modules are retained and corrected\n           with a residual CeNN (base(x) + scale*liquid(x)).\n    The exporter must save each variant to a DIFFERENT directory.\n    \"\"\"\n    root=Path(model_dir).resolve()\n    info=json.loads((root/\"config.json\").read_text(encoding=\"utf-8\"))\n    if info.get(\"format\")!=\"laya-v56-liquid-cenn-dual-v1\":\n        raise ValueError(\"Not a V5.6 dual-variant checkpoint\")\n    variant=str(info[\"variant\"]).lower()\n    if variant not in (\"full\",\"delta\"):\n        raise ValueError(f\"Invalid V5.6 variant: {variant}\")\n    cfg=AutoConfig.from_pretrained(str(root/\"encoder\"),local_files_only=True)\n    model=LayaStandalone(cfg,budget=int(info[\"head_context_budget\"]),\n                         head_layers=int(info[\"head_layers\"]))\n    for spec in info[\"compressed_projections\"]:\n        replace_projection(model,spec,initialized=False)\n    cells=list(info[\"liquid_cells\"])\n    if len(cells)!=5 or len({int(s[\"layer\"]) for s in cells})!=5:\n        raise ValueError(\"V5.6 models require precisely five unique CeNN layers\")\n    required_mode=\"replace\" if variant==\"full\" else \"residual\"\n    for cell in cells:\n        if cell[\"mode\"]!=required_mode:\n            raise ValueError(\"Mixed liquid modes are not a V5.6 variant\")\n        install_liquid_ffn(model,int(cell[\"layer\"]),\n            width=int(cell[\"width\"]),steps=int(cell[\"steps\"]),\n            mode=required_mode,dt_bound=float(cell[\"dt_bound\"]))\n    full=load_file(str(root/\"model.safetensors\"),device=\"cpu\")\n    model.load_state_dict(full,strict=True)\n    actual=find_liquid_cells(model)\n    assert len(actual)==5\n    assert all(m.mode==required_mode for _,m in actual)\n    if variant==\"full\":\n        assert all(m.base is None for _,m in actual)\n    else:\n        assert all(m.base is not None for _,m in actual)\n    model.to(device).eval().requires_grad_(False)\n    model.encoder.config.reference_compile=False\n    if compile_head:\n        model.head=torch.compile(model.head,mode=\"reduce-overhead\",dynamic=False)\n    tokenizer=AutoTokenizer.from_pretrained(\n        str(root/\"tokenizer\"),local_files_only=True)\n    return model,tokenizer\n"
runtime_file=OUT/"modeling_laya_v56.py"
runtime_file.write_text(V56_RUNTIME_CODE,encoding="utf-8")
spec=importlib.util.spec_from_file_location("v56_runtime",str(runtime_file))
assert spec and spec.loader
v56=importlib.util.module_from_spec(spec)
spec.loader.exec_module(v56)
normalize_question=v56.normalize_question
build_sequence=v56.build_sequence
collate_items=v56.collate_items
QTYPES=v56.QTYPES
teacher,_=v56.load_model(parent_dir,device="cuda")
student,_=v56.load_model(parent_dir,device="cuda")
teacher.eval().requires_grad_(False)
student.eval().requires_grad_(False)
teacher.encoder.config.reference_compile=False
student.encoder.config.reference_compile=False
native_unique=v56.unique_parameter_count(teacher)
assert len(student.encoder.layers)==28
assert len(v56.all_encoder_slots(student))==112
# Short liquid cell unit test: residual starts IDENTICAL and true replacement
# physically omits the old base module. Tests B,T,D shapes and gradients.
with torch.enable_grad():
    simple=nn.Linear(80,16,bias=False).to(DEVICE)
    sample=torch.randn(2,11,80,device=DEVICE)
    residual_cell=v56.LiquidCeNNProjection(
        simple,width=8,steps=2,mode="residual").to(DEVICE)
    torch.testing.assert_close(residual_cell(sample),simple(sample),atol=1e-6,rtol=1e-6)
    replacement_cell=v56.LiquidCeNNProjection(
        simple,width=8,steps=2,mode="replace").to(DEVICE)
    assert replacement_cell.base is None
    assert not any(name.startswith("base.") for name in replacement_cell.state_dict())
    result=replacement_cell(sample)
    assert result.shape==(2,11,16) and bool(torch.isfinite(result).all())
    result.square().mean().backward()
    assert replacement_cell.drive.weight.grad is not None
    del simple,sample,residual_cell,replacement_cell,result
gc.collect()
print("PASS: Liquid-CeNN residual identity, replacement weight removal and gradients")


In [ ]:
#@title 4. CASE-DISJOINT TRAIN / PROBE / AUDIT benchmark from V4.6
def _jsonish(x):
    if isinstance(x, str):
        try:
            return json.loads(x)
        except Exception:
            return x
    return x

def _canon_label(x):
    s = str(x).strip().lower()
    try:
        v = float(s)
        if math.isfinite(v) and v.is_integer():
            return str(int(v))
    except Exception:
        pass
    return s

def _gold_metadata(q, gold_answer):
    if not isinstance(gold_answer, dict):
        raise ValueError("missing gold answer")
    probs_raw = _jsonish(gold_answer.get("probabilities", {}))
    if not isinstance(probs_raw, dict) or len(probs_raw) < 2:
        raise ValueError("gold probabilities missing or malformed")

    gold_keys = [str(k) for k in probs_raw.keys()]
    if q["t"] == "choice":
        labels = [str(k) for k in q["crit"].keys()]
    elif q["t"] == "noul":
        labels = ["false", "true"]
    else:
        labels = list(gold_keys)
        if all(_canon_label(k).lstrip("-").isdigit() for k in labels):
            labels = sorted(labels, key=lambda k: int(_canon_label(k)))

    lookup = {_canon_label(k): float(v) for k, v in probs_raw.items()}
    if len(labels) != len(gold_keys) or any(_canon_label(k) not in lookup for k in labels):
        raise ValueError("Gold labels do not match encoded option order")

    gold_probs = np.asarray([lookup[_canon_label(k)] for k in labels], dtype=np.float64)
    gold_probs = np.clip(gold_probs, 0.0, None)
    if not np.isfinite(gold_probs).all() or gold_probs.sum() <= 0:
        raise ValueError("invalid gold probability vector")
    gold_probs /= gold_probs.sum()

    gold_label = _canon_label(gold_answer.get("label", labels[int(gold_probs.argmax())]))
    canon_labels = [_canon_label(k) for k in labels]
    if gold_label not in canon_labels:
        raise ValueError(f"gold label {gold_label!r} not in {labels!r}")
    return labels, gold_probs.tolist(), canon_labels.index(gold_label)

def make_items_from_rows(rows, seed, shuffle_items):
    out = []
    head_max_len = int(source_cfg.get("head_max_len", 192))
    for row in rows:
        state = _jsonish(row["state"])
        questions = _jsonish(row["questions"])
        gold = _jsonish(row["gold"])
        case_id = str(row.get("id", ""))
        workflow = str(row.get("workflow", "unknown"))
        state_text = state if isinstance(state, str) else json.dumps(state, ensure_ascii=False)
        state_ids = tokenizer(
            state_text.replace(tokenizer.mask_token, " "),
            add_special_tokens=False,
        )["input_ids"]

        for qid, qdef in questions.items():
            q = normalize_question(qdef)
            ids, markers = build_sequence(
                tokenizer,
                state,
                q,
                max_len=min(MAX_LEN, int(source_cfg.get("max_len", MAX_LEN))),
                head_max_len=head_max_len,
                truncate_left=isinstance(state, list),
                state_ids=state_ids,
            )
            if len(markers) < 2:
                continue
            labels, gold_probs, gold_index = _gold_metadata(q, gold[qid])
            if len(labels) != len(markers):
                raise ValueError(
                    f"{case_id}/{qid}: {len(markers)} markers vs {len(labels)} gold labels"
                )
            out.append({
                "ids": ids,
                "markers": markers,
                "qtype": QTYPES[q["t"]],
                "qtype_name": q["t"],
                "qid": str(qid),
                "case_id": case_id,
                "workflow": workflow,
                "labels": labels,
                "gold_probs": gold_probs,
                "gold_index": int(gold_index),
            })
    if shuffle_items:
        random.Random(seed).shuffle(out)
    return out

# Preserve V5.0's exact case-level TRAIN and PROBE split to reproduce its
# 120/120 step-120 finding. Use the DATASET'S DISTINCT TEST SPLIT for AUDIT.
# Reuse historical V5.1 train/PROBE for comparability, but SELECT using
# unused second-half DEV cases. TEST split stays unseen until after freezing.
rows=list(load_dataset("LocalLLaMA/typed-decisions","all",split="train"))
random.Random(SEED).shuffle(rows)
assert TRAIN_CASES+DEV_CASES<=len(rows)
train_rows=rows[:TRAIN_CASES]
historical_probe_rows=rows[TRAIN_CASES:TRAIN_CASES+DEV_CASES//2]
gold_val_rows=rows[TRAIN_CASES+DEV_CASES//2:TRAIN_CASES+DEV_CASES]
train_items=make_items_from_rows(train_rows,SEED,True)
probe_items=make_items_from_rows(historical_probe_rows,SEED+1,True)
gold_val_items=make_items_from_rows(gold_val_rows,SEED+2,True)
# Case namespaces protect against overlapping local split ids.
test_rows=list(load_dataset("LocalLLaMA/typed-decisions","all",split="test"))
random.Random(SEED+9001).shuffle(test_rows)
audit_rows=[dict(row,id="TEST::"+str(row.get("id",i)))
            for i,row in enumerate(test_rows)]
audit_items=make_items_from_rows(audit_rows,SEED+9001,True)
assert len(probe_items)>=HISTORICAL_PROBE_DECISIONS
assert len(gold_val_items)>=GOLD_VAL_DECISIONS
assert len(audit_items)>=AUDIT_DECISIONS
groups=[{x["case_id"] for x in seq} for seq in
        (train_items,probe_items,gold_val_items,audit_items)]
for i in range(len(groups)):
    for j in range(i+1,len(groups)):
        assert groups[i].isdisjoint(groups[j]),(i,j)
def direct_batch(selected):
    b=collate_items(selected,int(tokenizer.pad_token_id))
    return {k:v.to(device) for k,v in b.items()}
def batch(items,offset,n=BATCH):
    return direct_batch([items[(offset+i)%len(items)] for i in range(n)])
print("TRAIN:",len(train_items),"OLD PROBE:",len(probe_items),
      "NEW GOLD VAL:",len(gold_val_items),"FINAL TEST:",len(audit_items))


In [ ]:
#@title 5. Accuracy, divergence and paired CUDA/wall timing
@torch.inference_mode()
def probs_for(model,b):
    with torch.autocast("cuda",dtype=amp_dtype):
        scores,_=model(**b)
    return torch.softmax(scores.float().masked_fill(
        ~b["marker_mask"].bool(),-1e9),dim=-1).cpu()

@torch.inference_mode()
def get_reference(items,limit):
    out=[]
    for start in range(0,min(limit,len(items)),BATCH):
        chosen=items[start:min(start+BATCH,limit)]
        p=probs_for(teacher,direct_batch(chosen))
        for j,item in enumerate(chosen):
            out.append(dict(probs=p[j],teacher=int(p[j].argmax()),
                gold=int(item["gold_index"]),case=item["case_id"],qid=item["qid"]))
    return out

@torch.inference_mode()
def quality(model,items,ref):
    agree=correct=tcorrect=0;js=0.
    n=len(ref)
    for start in range(0,n,BATCH):
        chosen=items[start:min(n,start+BATCH)]
        p=probs_for(model,direct_batch(chosen))
        for j,item in enumerate(chosen):
            t=ref[start+j]["probs"]; s=p[j]
            assert s.shape==t.shape
            m=(s+t)/2
            js+=float(.5*((s*(s.clamp_min(1e-8).log()-m.clamp_min(1e-8).log())).sum()+
                            (t*(t.clamp_min(1e-8).log()-m.clamp_min(1e-8).log())).sum()))
            guess=int(s.argmax())
            agree+=int(guess==ref[start+j]["teacher"])
            correct+=int(guess==item["gold_index"])
            tcorrect+=int(ref[start+j]["teacher"]==item["gold_index"])
    return dict(decisions=n,teacher_agreement=agree/n,gold_accuracy=correct/n,
        teacher_gold_accuracy=tcorrect/n,gold_drop=(tcorrect-correct)/n,mean_js=js/n)

BENCH_BATCHES=[batch(probe_items,0,BATCH),batch(probe_items,11,BATCH)]
@torch.inference_mode()
def benchmark_paired(reference,candidate,repeats=BENCH_REPEATS):
    """Alternating A/B/B/A with CUDA events AND synchronized wall-clock timings.
    All timings exclude Colab compilation, dataset loading and model downloading."""
    results=[]
    for j,b in enumerate(BENCH_BATCHES):
        def once(m):
            torch.cuda.synchronize()
            a=torch.cuda.Event(enable_timing=True)
            z=torch.cuda.Event(enable_timing=True)
            before=time.perf_counter()
            a.record()
            with torch.autocast("cuda",dtype=amp_dtype):
                outputs=m(**b)
            z.record();z.synchronize()
            wall=(time.perf_counter()-before)*1000
            return float(a.elapsed_time(z)),float(wall)
        for k in range(4):
            once(reference);once(candidate)
        d={"reference":[],"candidate":[]}
        w={"reference":[],"candidate":[]}
        for k in range(repeats):
            order=("reference","candidate","candidate","reference") if k%2==0 else (
                "candidate","reference","reference","candidate")
            for name in order:
                gpu,cpu=once(reference if name=="reference" else candidate)
                d[name].append(gpu);w[name].append(cpu)
        ref=float(np.median(d["reference"]))
        cand=float(np.median(d["candidate"]))
        rw=float(np.median(w["reference"]))
        cw=float(np.median(w["candidate"]))
        results.append(dict(shape=j,batch=int(b["input_ids"].shape[0]),
            tokens=int(b["input_ids"].shape[1]),
            native_gpu_ms=ref,candidate_gpu_ms=cand,gpu_speedup=ref/max(cand,1e-9),
            native_wall_ms=rw,candidate_wall_ms=cw,wall_speedup=rw/max(cw,1e-9),
            native_p95_ms=float(np.percentile(d["reference"],95)),
            candidate_p95_ms=float(np.percentile(d["candidate"],95)),n=len(d["reference"])))
    return dict(median_gpu_speedup=float(np.median([r["gpu_speedup"] for r in results])),
        min_gpu_speedup=float(min(r["gpu_speedup"] for r in results)),
        median_wall_speedup=float(np.median([r["wall_speedup"] for r in results])),
        min_wall_speedup=float(min(r["wall_speedup"] for r in results)),
        per_shape=results)

# Optional model-quality helpers (aligned to V5.6 configuration).
def valid_quality(q):
    return (q["teacher_agreement"]>=MIN_TEACHER_AGREEMENT and
            q["gold_drop"]<=0.02 and
            q["mean_js"]<=MAX_TEST_JS)
def valid_speed(s):
    return (s["min_gpu_speedup"]>=MIN_SPEED_RATIO and
            s["min_wall_speedup"]>=MIN_SPEED_RATIO)



In [ ]:
#@title 6. Reconstruct exact 20-SVD compressed baseline for both variants
# This exact ordered list comes from the user's V5.0 greedy_combinations.json.
# It replaces the 520-candidate rank sweep; all unchanged projections are
# restored from PINNED pretrained V4.6 weights.
SELECTED_SPECS = [
  {
    "layer": 26,
    "leader": 25,
    "block": "attn",
    "component": "Wqkv",
    "mode": "pair_delta",
    "rank": 64
  },
  {
    "layer": 26,
    "leader": 25,
    "block": "attn",
    "component": "Wo",
    "mode": "pair_delta",
    "rank": 64
  },
  {
    "layer": 17,
    "leader": 16,
    "block": "attn",
    "component": "Wo",
    "mode": "pair_delta",
    "rank": 256
  },
  {
    "layer": 20,
    "leader": 19,
    "block": "attn",
    "component": "Wo",
    "mode": "pair_delta",
    "rank": 256
  },
  {
    "layer": 27,
    "block": "attn",
    "component": "Wqkv",
    "mode": "own_svd",
    "rank": 64
  },
  {
    "layer": 24,
    "block": "mlp",
    "component": "Wi",
    "mode": "own_svd",
    "rank": 256
  },
  {
    "layer": 23,
    "block": "attn",
    "component": "Wqkv",
    "mode": "own_svd",
    "rank": 256
  },
  {
    "layer": 27,
    "block": "attn",
    "component": "Wo",
    "mode": "own_svd",
    "rank": 32
  },
  {
    "layer": 23,
    "block": "attn",
    "component": "Wo",
    "mode": "own_svd",
    "rank": 32
  },
  {
    "layer": 23,
    "block": "mlp",
    "component": "Wo",
    "mode": "own_svd",
    "rank": 256
  },
  {
    "layer": 25,
    "block": "mlp",
    "component": "Wo",
    "mode": "own_svd",
    "rank": 256
  },
  {
    "layer": 24,
    "block": "mlp",
    "component": "Wo",
    "mode": "own_svd",
    "rank": 64
  },
  {
    "layer": 26,
    "block": "mlp",
    "component": "Wo",
    "mode": "own_svd",
    "rank": 256
  },
  {
    "layer": 21,
    "block": "mlp",
    "component": "Wo",
    "mode": "own_svd",
    "rank": 256
  },
  {
    "layer": 21,
    "block": "mlp",
    "component": "Wi",
    "mode": "own_svd",
    "rank": 256
  },
  {
    "layer": 20,
    "block": "mlp",
    "component": "Wo",
    "mode": "own_svd",
    "rank": 256
  },
  {
    "layer": 8,
    "block": "attn",
    "component": "Wo",
    "mode": "own_svd",
    "rank": 256
  },
  {
    "layer": 3,
    "block": "attn",
    "component": "Wo",
    "mode": "own_svd",
    "rank": 256
  },
  {
    "layer": 2,
    "block": "attn",
    "component": "Wo",
    "mode": "own_svd",
    "rank": 256
  },
  {
    "layer": 15,
    "block": "attn",
    "component": "Wo",
    "mode": "own_svd",
    "rank": 256
  }
]
assert len(SELECTED_SPECS)==20
assert len({(s["layer"],s["block"],s["component"]) for s in SELECTED_SPECS})==20
assert sum(s["mode"]=="pair_delta" for s in SELECTED_SPECS)==4
# Stable fingerprint of the exact 20 selected projections and ranks.
# Used in model config, provenance and release verification.
spec_hash=hashlib.sha256(
    json.dumps(SELECTED_SPECS,sort_keys=True,separators=(",",":")).encode("utf-8")
).hexdigest()
assert len(spec_hash)==64
print("Compression specification SHA-256:",spec_hash)
def family_key(f):
    return (int(f["layer"]),str(f["block"]),str(f["component"]),str(f["mode"]))
def deterministic_factors(f):
    family_seed=SEED+sum((i+1)*ord(ch) for i,ch in enumerate(
        ":".join(map(str,family_key(f)))))
    with torch.random.fork_rng(devices=[torch.cuda.current_device()]):
        torch.manual_seed(family_seed)
        if f["mode"]=="own_svd":
            return v56.own_svd_factors(
                v56.projection_at(student,f),
                max_rank=256,oversampling=8,niter=2)[:2]
        return v56.svd_factors(
            student,int(f["leader"]),int(f["layer"]),
            str(f["component"]),max_rank=256,oversampling=8,niter=2)[:2]

PROBE_REF=get_reference(probe_items,HISTORICAL_PROBE_DECISIONS)
GOLD_VAL_REF=get_reference(gold_val_items,GOLD_VAL_DECISIONS)

native_q=quality(student,probe_items,PROBE_REF)
assert native_q["teacher_agreement"]==1.0,native_q
for j,s in enumerate(SELECTED_SPECS,1):
    factors=deterministic_factors(s)
    old=v56.replace_projection(student,s,factors=factors,initialized=True)
    assert v56.projection_at(student,s) is not old
    print(f"Selected {j:02d}/{len(SELECTED_SPECS)}: layer={s['layer']} "
          f"{s['block']}.{s['component']} {s['mode']} rank={s['rank']}")
unique_student=v56.unique_parameter_count(student)
estimated_bytes=(native_unique-unique_student)*4
assert estimated_bytes>0
initial_probe=quality(student,probe_items,PROBE_REF)
initial_gold_validation=quality(student,gold_val_items,GOLD_VAL_REF)
print("Historical initial PROBE (not used for selection):",initial_probe)
print("New gold-validation baseline:",initial_gold_validation)
print("Teacher gold accuracy in gold-validation:",initial_gold_validation["teacher_gold_accuracy"])

print("Projected parameter savings:",f"{estimated_bytes:,} bytes",
      f"({estimated_bytes/1e6:.3f} MB)")
(OUT/"selected_projections.json").write_text(json.dumps({
    "source_repo":TEACHER_REPO,"source_revision":PINNED_TEACHER_REVISION,
    "specifications":SELECTED_SPECS,"estimated_saved_fp32_bytes":estimated_bytes,
    "initial_probe":initial_probe},indent=2),encoding="utf-8")

# Optional exact, already-gold-tuned V5.2 standalone checkpoint. The config
# must contain the SAME 20 selected projections in the SAME order.
BASELINE_SOURCE="reconstructed_V5.1_frozen_head"
if BASELINE_V52_STANDALONE_DIR.strip():
    root=Path(BASELINE_V52_STANDALONE_DIR.strip()).resolve()
    cfg=json.loads((root/"config.json").read_text(encoding="utf-8"))
    assert cfg["compressed_projections"]==SELECTED_SPECS
    assert cfg["source_revision"]==PINNED_TEACHER_REVISION
    tensors=load_file(str(root/"model.safetensors"),device="cpu")
    student.load_state_dict(tensors,strict=True)
    BASELINE_SOURCE=str(root)
student.eval().requires_grad_(False)
BASELINE_VAL=quality(student,gold_val_items,GOLD_VAL_REF)
BASELINE_OLD_PROBE=quality(student,probe_items,PROBE_REF)
print("CONTROL BASELINE SOURCE:",BASELINE_SOURCE)
print("CONTROL GOLD VALIDATION:",BASELINE_VAL)
(OUT/"baseline_metrics.json").write_text(json.dumps(dict(
    source=BASELINE_SOURCE,teacher_revision=PINNED_TEACHER_REVISION,
    gold_validation=BASELINE_VAL,historical_probe=BASELINE_OLD_PROBE,
    compressed_spec_hash=spec_hash),indent=2))

# IMPORTANT: clone the baseline from the exact compressed state using strict
# state_dict loading (no repeated randomized SVD!). This is used for the final
# paired GPU benchmark and ensures a fair baseline.
baseline_model,_=v56.load_model(parent_dir,device="cuda")
baseline_model.encoder.config.reference_compile=False
baseline_model.eval().requires_grad_(False)
for spec in SELECTED_SPECS:
    v56.replace_projection(baseline_model,spec,initialized=False)
baseline_model.load_state_dict(student.state_dict(),strict=True)
baseline_model.eval().requires_grad_(False)
BASELINE_UNIQUE=v56.unique_parameter_count(baseline_model)
assert BASELINE_UNIQUE==v56.unique_parameter_count(student)
baseline_recheck=quality(baseline_model,gold_val_items,GOLD_VAL_REF)
# The same weights should give the same choices; allow tiny floating-point
# changes in probabilistic divergence on different GPU kernels.
assert baseline_recheck["gold_accuracy"]==BASELINE_VAL["gold_accuracy"]
assert baseline_recheck["teacher_agreement"]==BASELINE_VAL["teacher_agreement"]
assert abs(baseline_recheck["mean_js"]-BASELINE_VAL["mean_js"])<1e-5
print("PASS: compressed baseline exactly reproduced without SVD recomputation")


In [ ]:
#@title 7. Shared five-layer train, evaluation, export and release helpers
# Models are trained sequentially, never share trainable CeNN weights, and
# are separately checked against the SAME frozen compressed baseline.
from collections import OrderedDict
import traceback
from safetensors.torch import save_file,load_file
from pathlib import Path

FULL_DIR=OUT/"FULL_standalone_model"
DELTA_DIR=OUT/"DELTA_standalone_model"
FULL_DIR.mkdir(parents=True,exist_ok=True)
DELTA_DIR.mkdir(parents=True,exist_ok=True)
VARIANT_RESULTS={}
BENCH_ORIGINAL=BENCH_BATCHES
TRAIN_ORDER_SEED=SEED+560
BASELINE_TEST_REF=None

def variant_liquid_specs(variant):
    mode="replace" if variant=="FULL" else "residual"
    return [dict(s,mode=mode) for s in SELECTED_LIQUID_SPECS]

def install_five(model,variant):
    installed=[]
    for spec in variant_liquid_specs(variant):
        idx=int(spec["layer"])
        old=v56.install_liquid_ffn(model,idx,width=int(spec["width"]),
            steps=int(spec["steps"]),mode=spec["mode"],
            dt_bound=float(spec["dt_bound"]))
        installed.append((idx,old))
    found=v56.find_liquid_cells(model)
    assert len(found)==5
    if variant=="FULL":
        assert all(cell.base is None for _,cell in found)
    else:
        assert all(cell.base is not None for _,cell in found)
        assert all(cell.scale.item()==0. for _,cell in found)
    return installed

def activation_capture(model,layer):
    old=model.encoder.layers[int(layer)].mlp.Wo
    captured=[]
    def hook(_module,args,out):
        captured.append((args[0].detach().float().cpu().to(torch.float16),
                         out.detach().float().cpu().to(torch.float16)))
    h=old.register_forward_hook(hook)
    try:
        for k in range(CAPTURE_BATCHES):
            bx=batch(train_items,(k*BATCH+layer*11)%len(train_items),BATCH)
            with torch.no_grad(),torch.autocast("cuda",dtype=amp_dtype):
                model(**bx)
    finally:
        h.remove()
    assert len(captured)==CAPTURE_BATCHES
    return captured

def fit_full_cell(model,spec,samples):
    old=model.encoder.layers[int(spec["layer"])].mlp.Wo
    candidate=v56.LiquidCeNNProjection(old,
        width=int(spec["width"]),steps=int(spec["steps"]),
        mode="replace",dt_bound=float(spec["dt_bound"])).to(
        device=DEVICE,dtype=torch.float32)
    optimizer=torch.optim.AdamW(candidate.parameters(),
        lr=LOCAL_DISTILL_LR,weight_decay=0.)
    trace=[]
    for step in range(1,LOCAL_DISTILL_STEPS+1):
        xx,yy=samples[(step-1)%len(samples)]
        x=xx.to(device=DEVICE,dtype=torch.float32)
        y=yy.to(device=DEVICE,dtype=torch.float32)
        candidate.train()
        optimizer.zero_grad(set_to_none=True)
        predicted=candidate(x)
        mse=F.mse_loss(predicted,y)/y.square().mean().clamp_min(1e-7)
        if not bool(torch.isfinite(mse)):
            raise RuntimeError(f"Non-finite local fit for layer {spec['layer']}")
        mse.backward()
        torch.nn.utils.clip_grad_norm_(candidate.parameters(),1.)
        optimizer.step()
        if step in (1,LOCAL_DISTILL_STEPS) or step%20==0:
            trace.append(dict(step=step,relative_mse=float(mse.detach().cpu())))
    candidate.eval()
    model.encoder.layers[int(spec["layer"])].mlp.Wo=candidate
    original_params=sum(p.numel() for p in old.parameters())
    cell_params=sum(p.numel() for p in candidate.parameters())
    assert cell_params<original_params,(
        "FULL should actually remove parameters",spec,original_params,cell_params)
    del optimizer
    return dict(layer=spec["layer"],width=spec["width"],steps=spec["steps"],
        original_projection_parameters=original_params,
        liquid_parameters=cell_params,
        saved_projection_parameters=original_params-cell_params,
        local_imitation_loss=trace)

def gold_training_batch(rng,order,cursor):
    chosen=[]
    for _ in range(TRAIN_B):
        if cursor>=len(order):
            rng.shuffle(order)
            cursor=0
        chosen.append(train_items[order[cursor]])
        cursor+=1
    b=direct_batch(chosen)
    assert bool((b["marker_pos"][b["marker_mask"]]<TRAIN_MAX_TOKENS).all())
    for key in ("input_ids","attention_mask"):
        b[key]=b[key][:,:TRAIN_MAX_TOKENS]
    hard=torch.tensor([x["gold_index"] for x in chosen],
        dtype=torch.long,device=DEVICE)
    soft=torch.zeros(b["marker_mask"].shape,dtype=torch.float32,device=DEVICE)
    for j,item in enumerate(chosen):
        p=torch.tensor(item["gold_probs"],dtype=torch.float32,device=DEVICE)
        assert len(p)==len(item["markers"])
        soft[j,:len(p)]=p/p.sum().clamp_min(1e-8)
    return b,hard,soft,cursor

def fine_tune_variant(model,variant):
    model.requires_grad_(False)
    cells=v56.find_liquid_cells(model)
    assert len(cells)==5
    for _,cell in cells:
        for name,p in cell.named_parameters():
            if not name.startswith("base."):
                p.requires_grad_(True)
    trainable=[p for p in model.parameters() if p.requires_grad]
    assert trainable
    scales=[cell.scale for _,cell in cells] if variant=="DELTA" else []
    non_scales=[p for p in trainable if id(p) not in {id(s) for s in scales}]
    optim_groups=(
      [dict(params=non_scales,lr=DELTA_LR),
       dict(params=scales,lr=DELTA_SCALE_LR)]
      if variant=="DELTA" else [dict(params=trainable,lr=FULL_LR)])
    optimizer=torch.optim.AdamW(optim_groups,weight_decay=0.)
    scaler=torch.amp.GradScaler("cuda",enabled=amp_dtype==torch.float16)
    model.eval()
    rng=random.Random(TRAIN_ORDER_SEED)
    order=list(range(len(train_items)))
    rng.shuffle(order)
    cursor=0
    state_names={k for k,p in model.named_parameters() if p.requires_grad}
    best=quality(model,gold_val_items,GOLD_VAL_REF)
    best_step=0
    log=[]
    best_path=OUT/(variant+"_best_liquid.safetensors")
    def save_local(step,q):
        train_state=OrderedDict((k,p.detach().cpu().contiguous().clone())
            for k,p in model.named_parameters() if p.requires_grad)
        save_file(train_state,str(best_path),metadata={
            "format":"pt","variant":variant,"step":str(step)})
        (OUT/(variant+"_best_liquid.json")).write_text(
            json.dumps(dict(step=step,validation=q,variant=variant),indent=2))
    save_local(0,best)
    max_steps=FULL_TRAIN_STEPS if variant=="FULL" else DELTA_TRAIN_STEPS
    patience=0
    for step in range(1,max_steps+1):
        bx,hard,soft,cursor=gold_training_batch(rng,order,cursor)
        with torch.no_grad(),torch.autocast("cuda",dtype=amp_dtype):
            teacher_scores,_=teacher(**bx)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast("cuda",dtype=amp_dtype):
            student_scores,_=model(**bx)
        valid=bx["marker_mask"].bool()
        s=student_scores.float().masked_fill(~valid,-1e9)
        t=teacher_scores.float().masked_fill(~valid,-1e9)
        logp=F.log_softmax(s,dim=-1)
        loss_gold=F.nll_loss(logp,hard)
        loss_soft=-(soft*logp).sum(-1).mean()
        kl=F.kl_div(F.log_softmax(s/2,dim=-1),
            F.softmax(t/2,dim=-1),reduction="batchmean")*4.
        loss=GOLD_CE_WEIGHT*loss_gold+GOLD_SOFT_WEIGHT*loss_soft+TEACHER_KL_WEIGHT*kl
        if not bool(torch.isfinite(loss)):
            raise RuntimeError(f"{variant} has non-finite loss at step {step}")
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(trainable,1.)
        scaler.step(optimizer)
        scaler.update()
        if step%TRAIN_CHECK_EVERY==0 or step==max_steps:
            q=quality(model,gold_val_items,GOLD_VAL_REF)
            improved=(q["gold_accuracy"],q["teacher_agreement"],-q["mean_js"])>(
                best["gold_accuracy"],best["teacher_agreement"],-best["mean_js"])
            if improved:
                best=q;best_step=step;patience=0
                save_local(step,q)
            else:
                patience+=1
            rec=dict(variant=variant,step=step,loss=float(loss.detach().cpu()),
                gold_loss=float(loss_gold.detach().cpu()),
                gold_soft=float(loss_soft.detach().cpu()),
                teacher_kl=float(kl.detach().cpu()),validation=q,
                improved=improved)
            log.append(rec)
            (OUT/(variant+"_training_progress.json")).write_text(
                json.dumps(log,indent=2))
            print(f"{variant}: step={step}, GOLD={q['gold_accuracy']:.3%}, "
                  f"teacher_match={q['teacher_agreement']:.3%}, selected={improved}")
            if patience>=5:
                print(variant,": gold validation plateau, early stop")
                break
    saved=load_file(str(best_path),device="cpu")
    assert set(saved)==state_names
    with torch.no_grad():
        for name,p in model.named_parameters():
            if name in saved:
                p.copy_(saved[name].to(device=p.device,dtype=p.dtype))
    model.eval().requires_grad_(False)
    final_val=quality(model,gold_val_items,GOLD_VAL_REF)
    assert final_val["gold_accuracy"]==best["gold_accuracy"]
    history=dict(variant=variant,selected_step=best_step,
      validation_at_start=log[0]["validation"] if log else best,
      final_validation=final_val,log=log,
      trained_parameter_count=sum(p.numel() for p in trainable))
    (OUT/(variant+"_training_summary.json")).write_text(json.dumps(history,indent=2))
    return history

def benchmark_variant(model):
    global BENCH_BATCHES,BASELINE_TEST_REF
    BENCH_BATCHES=[batch(audit_items,0,BATCH),batch(audit_items,11,BATCH)]
    speed=benchmark_paired(baseline_model,model,repeats=max(BENCH_REPEATS,15))
    BENCH_BATCHES=BENCH_ORIGINAL
    return speed

def evaluate_variant(model,variant):
    global BASELINE_TEST_REF
    assert not any(p.requires_grad for p in model.parameters())
    assert len(v56.find_liquid_cells(model))==5
    if BASELINE_TEST_REF is None:
        BASELINE_TEST_REF=get_reference(audit_items,AUDIT_DECISIONS)
    reference=BASELINE_TEST_REF
    baseline_gold=quality(baseline_model,audit_items,reference)
    metrics=quality(model,audit_items,reference)
    speed=benchmark_variant(model)
    report=dict(variant=variant,teacher_gold=metrics["teacher_gold_accuracy"],
        compressed_baseline_gold=baseline_gold["gold_accuracy"],
        variant_gold=metrics["gold_accuracy"],
        advantage_vs_teacher_pp=100*(metrics["gold_accuracy"]-
                                     metrics["teacher_gold_accuracy"]),
        advantage_vs_baseline_pp=100*(metrics["gold_accuracy"]-
                                     baseline_gold["gold_accuracy"]),
        model_test=metrics,baseline_test=baseline_gold,
        speed_against_baseline=speed)
    (OUT/(variant+"_test_report.json")).write_text(json.dumps(report,indent=2))
    print("TEST "+variant,report["variant_gold"],
          "BASELINE",report["compressed_baseline_gold"],
          "TEACHER",report["teacher_gold"])
    return report

def export_and_publish(model,variant,training,performance,progressive=None):
    assert variant in ("FULL","DELTA")
    root=FULL_DIR if variant=="FULL" else DELTA_DIR
    root.mkdir(parents=True,exist_ok=True)
    (root/"encoder").mkdir(exist_ok=True)
    (root/"tokenizer").mkdir(exist_ok=True)
    model.encoder.config.save_pretrained(str(root/"encoder"))
    tokenizer.save_pretrained(str(root/"tokenizer"))
    (root/"modeling_laya_v56.py").write_text(V56_RUNTIME_CODE,encoding="utf-8")
    info=dict(format="laya-v56-liquid-cenn-dual-v1",
       model_type="laya-v56-"+variant.lower()+"-five-liquid",
       variant=variant.lower(),
       head_context_budget=64,head_layers=int(source_config["head_layers"]),
       source_repo=TEACHER_REPO,source_revision=PINNED_TEACHER_REVISION,
       compressed_projections=SELECTED_SPECS,
       liquid_cells=variant_liquid_specs(variant),
       selected_step=training["selected_step"],
       output_scope="typed-decision option scores only; 28 encoder layers")
    (root/"config.json").write_text(json.dumps(info,indent=2))
    # These full binary models are intentionally NEVER added to statistics ZIP.
    weight_file=root/"model.safetensors"
    tensors=OrderedDict((k,v.detach().cpu().contiguous().clone())
                 for k,v in model.state_dict().items())
    save_file(tensors,str(weight_file),metadata={
        "format":"pt","variant":variant})
    del tensors
    gc.collect()
    baseline_bytes=4*v56.unique_parameter_count(baseline_model)
    actual_parameter_change_bytes=4*(
        v56.unique_parameter_count(baseline_model)-v56.unique_parameter_count(model))
    teacher_checkpoint_bytes=(parent_dir/"model.safetensors").stat().st_size
    checkpoint_bytes=weight_file.stat().st_size
    saved_from_teacher=teacher_checkpoint_bytes-checkpoint_bytes
    reloaded,_tok=v56.load_v56_model(root,device="cuda")
    bx=batch(gold_val_items,0,BATCH)
    with torch.no_grad(),torch.autocast("cuda",dtype=amp_dtype):
        a,_=model(**bx);b,_=reloaded(**bx)
    reloaded_matches=(bool(torch.equal(a.argmax(-1),b.argmax(-1)))
       and bool(torch.allclose(a.float(),b.float(),atol=2e-3,rtol=2e-3)))
    reload_max_diff=float((a.float()-b.float()).abs().max())
    del reloaded
    gc.collect();torch.cuda.empty_cache()
    q=performance["model_test"]
    base=performance["baseline_test"]
    speed=performance["speed_against_baseline"]
    gates=dict(
       exactly_five_liquid_ffns=len(v56.find_liquid_cells(model))==5,
       no_gold_validation_regression=
          training["final_validation"]["gold_accuracy"]+1e-9>=BASELINE_VAL["gold_accuracy"],
       no_test_gold_regression=q["gold_accuracy"]+1e-9>=base["gold_accuracy"],
       test_teacher_agreement=q["teacher_agreement"]>=MIN_TEACHER_AGREEMENT,
       test_prediction_divergence=q["mean_js"]<=MAX_TEST_JS,
       full_model_saves_at_least_100MB=saved_from_teacher>=MIN_TOTAL_SAVED_BYTES,
       measured_latency_within_10pct=(
          speed["min_gpu_speedup"]>=MIN_SPEED_RATIO and
          speed["min_wall_speedup"]>=MIN_SPEED_RATIO),
       offline_model_reload_parity=reloaded_matches)
    if variant=="FULL":
        gates["full_reduces_parameters_vs_compressed"]=(actual_parameter_change_bytes>=MIN_FULL_SAVED_VS_BASELINE_BYTES)
    else:
        gates["delta_keeps_original_projection"]=all(
            cell.base is not None for _,cell in v56.find_liquid_cells(model))
    gates["all_pass"]=all(gates.values())
    report=dict(version="V5.6",variant=variant,liquid_cell_specs=variant_liquid_specs(variant),
      train=training,test=performance,progressive_distillation=progressive,
      baseline_parameter_bytes=baseline_bytes,student_model_bytes=checkpoint_bytes,
      teacher_model_bytes=teacher_checkpoint_bytes,saved_from_teacher_bytes=saved_from_teacher,
      theoretical_parameter_bytes_saved_vs_baseline=actual_parameter_change_bytes,
      offline_reload_max_abs_diff=reload_max_diff,gates=gates,
      caveat="TEST dataset is reused across earlier V5.x experiments. Test accuracy differences of 1-3 examples are not statistically conclusive.")
    (root/("v56_"+variant.lower()+"_report.json")).write_text(
        json.dumps(report,indent=2))
    hf_repo=FULL_HF_REPO if variant=="FULL" else DELTA_HF_REPO
    readme=f"""---
library_name: pytorch
base_model: {TEACHER_REPO}
tags:
- modernbert
- liquid-cenn
- {variant.lower()}-variant
- model-compression
- experimental
---

# Laya V5.6 — {variant} Liquid CeNN (five FFN output layers)

Variant **{variant}** replaces FFN Wo completely (FULL) or adds learned
liquid residuals to original compressed Wo (DELTA) at layers
**20, 24, 23, 26 and 8**. All original Transformer attentions remain.

- Teacher gold TEST: **{q['teacher_gold_accuracy']:.3%}**
- Compressed baseline gold TEST: **{base['gold_accuracy']:.3%}**
- {variant} gold TEST: **{q['gold_accuracy']:.3%}**
- Gold validation: **{training['final_validation']['gold_accuracy']:.3%}**
- Teacher top-1 agreement: **{q['teacher_agreement']:.3%}**
- Median GPU speed ratio vs baseline: **{speed['median_gpu_speedup']:.3f}x**
- Original teacher checkpoint bytes: **{teacher_checkpoint_bytes:,}**
- Standalone {variant} FP32 checkpoint bytes: **{checkpoint_bytes:,}**
- Savings relative to teacher: **{saved_from_teacher:,} bytes**
- Change vs 20-SVD compressed baseline: **{actual_parameter_change_bytes:+,} theoretical FP32 bytes saved** (negative means EXTRA weights)
- Release gates passed: **{gates['all_pass']}**

A ratio <1.0 means slower than the compressed baseline on tested shapes.
This is experimental, with a reused small test set, and not evidence that
all encoder layers or self-attention were replaced. The DELTA design
preserves its existing FFN projections and increases their parameters.

## Offline load
\`\`\`python
from modeling_laya_v56 import load_v56_model
model,tokenizer=load_v56_model("/local/model",device="cuda")
\`\`\`
"""
    (root/"README.md").write_text(readme)
    (root/"requirements.txt").write_text(
       "torch>=2.4\ntransformers==4.57.6\nsafetensors>=0.4\nhuggingface_hub>=0.25,<1\n")
    hf_status="blocked_by_quality_gates"
    hf_error=None
    url=None
    if gates["all_pass"] and PUBLISH_TO_HF:
        try:
            publisher=HfApi()
            try: account=publisher.whoami()["name"]
            except Exception:
                notebook_login()
                account=publisher.whoami()["name"]
            assert account=="vtava",f"HF account was {account!r}, wanted vtava"
            if publisher.repo_exists(hf_repo,repo_type="model"):
                existing=set(publisher.list_repo_files(hf_repo,repo_type="model"))
                if existing-{".gitattributes",".gitignore"} and not ALLOW_UPDATE_EXISTING:
                    raise RuntimeError("Nonempty repo: explicit overwrite opt-in required")
            else:
                publisher.create_repo(hf_repo,repo_type="model",
                      private=PRIVATE_HF_REPO,exist_ok=False)
            publisher.upload_large_folder(repo_id=hf_repo,repo_type="model",
                  folder_path=str(root))
            actual=set(publisher.list_repo_files(hf_repo,repo_type="model"))
            wanted={"model.safetensors","config.json","modeling_laya_v56.py",
                    "encoder/config.json","README.md","requirements.txt"}
            assert wanted.issubset(actual),wanted-actual
            hf_status="published"
            url=f"https://huggingface.co/{hf_repo}"
            print("PUBLISHED",variant,url)
        except Exception as exc:
            hf_status="upload_error"
            hf_error=f"{type(exc).__name__}: {exc}"
            (OUT/(variant+"_hf_upload_error.txt")).write_text(traceback.format_exc())
            print(variant,"publication failed",hf_error)
    elif gates["all_pass"]:
        hf_status="user_disabled"
    status=dict(variant=variant,repo=hf_repo,status=hf_status,url=url,
           error=hf_error,gates=gates)
    (OUT/(variant+"_huggingface_status.json")).write_text(json.dumps(status,indent=2))
    report["huggingface"]=status
    (root/("v56_"+variant.lower()+"_report.json")).write_text(json.dumps(report,indent=2))
    VARIANT_RESULTS[variant]=dict(performance=performance,training=training,
       model_bytes=checkpoint_bytes,savings_vs_teacher=saved_from_teacher,
       delta_bytes_vs_compressed=actual_parameter_change_bytes,
       gates=gates,huggingface=status)
    print(variant,"RELEASE GATES",json.dumps(gates,indent=2))
    return report


In [ ]:
#@title 8. FULL FIRST: progressively replace 5 Wo projections and activation-distill
assert len(v56.find_liquid_cells(student))==0
progressive=[]
for spec in SELECTED_LIQUID_SPECS:
    samples=activation_capture(student,spec["layer"])
    fit=fit_full_cell(student,spec,samples)
    del samples
    q=quality(student,gold_val_items,GOLD_VAL_REF)
    progressive.append(dict(fit=fit,gold_validation=q,
       active_liquid_count=len(v56.find_liquid_cells(student))))
    print(f"FULL stage {len(progressive)}/5, layer={spec['layer']}, "
          f"gold={q['gold_accuracy']:.3%}, JS={q['mean_js']:.5f}")
    (OUT/"FULL_progressive_replacements.json").write_text(json.dumps(
        dict(baseline_validation=BASELINE_VAL,stages=progressive),indent=2))
    gc.collect()
assert len(v56.find_liquid_cells(student))==5
FULL_INITIAL_VAL=quality(student,gold_val_items,GOLD_VAL_REF)
print("FULL initial all-five validation:",FULL_INITIAL_VAL)


In [ ]:
#@title 9. FULL FIRST: jointly fine-tune, frozen TEST, standalone export, gated HF
full_training=fine_tune_variant(student,"FULL")
full_test=evaluate_variant(student,"FULL")
full_report=export_and_publish(student,"FULL",full_training,full_test,
    progressive=progressive)
# Release GPU memory before initializing completely independent DELTA model.
del student
gc.collect()
torch.cuda.empty_cache()
print("FULL experiment finished. Starting DELTA from exact compressed control.")


In [ ]:
#@title 10. DELTA SECOND: restore compressed model + add 5 zero-initialized residual liquid CeNNs
student,_=v56.load_model(parent_dir,device="cuda")
student.encoder.config.reference_compile=False
for spec in SELECTED_SPECS:
    v56.replace_projection(student,spec,initialized=False)
student.load_state_dict(baseline_model.state_dict(),strict=True)
student.eval().requires_grad_(False)
before_delta=quality(student,gold_val_items,GOLD_VAL_REF)
assert before_delta["gold_accuracy"]==BASELINE_VAL["gold_accuracy"]
assert before_delta["teacher_agreement"]==BASELINE_VAL["teacher_agreement"]
install_five(student,"DELTA")
DELTA_INITIAL_VAL=quality(student,gold_val_items,GOLD_VAL_REF)
# ZERO coefficient -> exact compressed baseline prediction before fine-tune.
assert DELTA_INITIAL_VAL["gold_accuracy"]==BASELINE_VAL["gold_accuracy"]
assert DELTA_INITIAL_VAL["teacher_agreement"]==BASELINE_VAL["teacher_agreement"]
assert abs(DELTA_INITIAL_VAL["mean_js"]-BASELINE_VAL["mean_js"])<1e-5
assert len(v56.find_liquid_cells(student))==5
print("PASS: Five DELTA CeNN modules, zero-init matches frozen compressed baseline")
(OUT/"DELTA_initial_parity.json").write_text(json.dumps(dict(
  initial=DELTA_INITIAL_VAL,baseline=BASELINE_VAL,
  layers=SELECTED_LIQUID_SPECS),indent=2))


In [ ]:
#@title 11. DELTA SECOND: gold fine-tuning, frozen TEST, separate gated HF repository
delta_training=fine_tune_variant(student,"DELTA")
delta_test=evaluate_variant(student,"DELTA")
delta_report=export_and_publish(student,"DELTA",delta_training,delta_test,
    progressive=None)
print("DELTA experiment finished. FULL and DELTA have independent checkpoints.")


In [ ]:
#@title 12. Compare FULL vs DELTA vs compressed control + original teacher
assert set(VARIANT_RESULTS)=={"FULL","DELTA"}
f=VARIANT_RESULTS["FULL"]
d=VARIANT_RESULTS["DELTA"]
b=f["performance"]["baseline_test"]["gold_accuracy"]
t=f["performance"]["model_test"]["teacher_gold_accuracy"]
assert b==d["performance"]["baseline_test"]["gold_accuracy"]
assert t==d["performance"]["model_test"]["teacher_gold_accuracy"]
comparison=dict(
 original_teacher_gold_accuracy=t,compressed_baseline_gold_accuracy=b,
 full_gold_accuracy=f["performance"]["model_test"]["gold_accuracy"],
 delta_gold_accuracy=d["performance"]["model_test"]["gold_accuracy"],
 full_gold_validation=f["training"]["final_validation"]["gold_accuracy"],
 delta_gold_validation=d["training"]["final_validation"]["gold_accuracy"],
 full_teacher_agreement=f["performance"]["model_test"]["teacher_agreement"],
 delta_teacher_agreement=d["performance"]["model_test"]["teacher_agreement"],
 full_GPU_speed_ratio=f["performance"]["speed_against_baseline"]["median_gpu_speedup"],
 delta_GPU_speed_ratio=d["performance"]["speed_against_baseline"]["median_gpu_speedup"],
 full_model_checkpoint_bytes=f["model_bytes"],
 delta_model_checkpoint_bytes=d["model_bytes"],
 full_saved_vs_teacher_bytes=f["savings_vs_teacher"],
 delta_saved_vs_teacher_bytes=d["savings_vs_teacher"],
 full_saved_vs_compressed_baseline_theoretical_bytes=f["delta_bytes_vs_compressed"],
 delta_saved_vs_compressed_baseline_theoretical_bytes=d["delta_bytes_vs_compressed"],
 full_hf=f["huggingface"],delta_hf=d["huggingface"],
 note="These variants were trained in fixed FULL-then-DELTA order. The reused 200-decision TEST does not prove statistically significant generalization gains.")
(OUT/"FULL_vs_DELTA_comparison.json").write_text(json.dumps(comparison,indent=2))
print(json.dumps(comparison,indent=2))


In [ ]:
#@title 13. AUTO-DOWNLOAD statistics-only ZIP; never download binary models
import zipfile,csv,io,hashlib,json
from pathlib import Path
from datetime import datetime,timezone
OUT=Path("/content/Laya_V56_FullThenDelta")
assert OUT.is_dir()
ZIP=OUT.parent/"Laya_V56_FULL_vs_DELTA_Statistics.zip"
summary={
 "experiment":"Laya V5.6, FULL then DELTA separate CeNN variants",
 "created_utc":datetime.now(timezone.utc).isoformat(),
 "results":globals().get("comparison",{}),
 "hf_full":globals().get("VARIANT_RESULTS",{}).get("FULL",{}).get("huggingface"),
 "hf_delta":globals().get("VARIANT_RESULTS",{}).get("DELTA",{}).get("huggingface"),
 "note":"No weights, adapters, tokenizer or checkpoint tensors included."
}
(OUT/"statistics_summary.json").write_text(json.dumps(summary,indent=2))
allowed={
 "baseline_metrics.json","selected_projections.json",
 "FULL_progressive_replacements.json","FULL_best_liquid.json",
 "FULL_training_progress.json","FULL_training_summary.json",
 "FULL_test_report.json","FULL_huggingface_status.json",
 "FULL_hf_upload_error.txt","DELTA_initial_parity.json",
 "DELTA_best_liquid.json","DELTA_training_progress.json",
 "DELTA_training_summary.json","DELTA_test_report.json",
 "DELTA_huggingface_status.json","DELTA_hf_upload_error.txt",
 "FULL_vs_DELTA_comparison.json","statistics_summary.json"
}
paths=[OUT/f for f in sorted(allowed) if (OUT/f).is_file()]
for v in ("FULL","DELTA"):
    modeldir=OUT/(v+"_standalone_model")
    for name in (f"v56_{v.lower()}_report.json","README.md","config.json"):
        p=modeldir/name
        if p.is_file():
            paths.append(p)
assert paths
manifest=dict(created_utc=datetime.now(timezone.utc).isoformat(),
 archive_type="metrics-only",files=[])
tab=io.StringIO()
writer=csv.writer(tab)
writer.writerow(("model","test_gold_accuracy","validation_gold_accuracy",
                 "gpu_speed_ratio","checkpoint_bytes","hf_status"))
for name in ("FULL","DELTA"):
    d=globals().get("VARIANT_RESULTS",{}).get(name)
    if d is not None:
        writer.writerow((name,d["performance"]["model_test"]["gold_accuracy"],
        d["training"]["final_validation"]["gold_accuracy"],
        d["performance"]["speed_against_baseline"]["median_gpu_speedup"],
        d["model_bytes"],d["huggingface"]["status"]))
csv_bytes=tab.getvalue().encode()
with zipfile.ZipFile(ZIP,"w",compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for p in paths:
        name=("model_metadata/"+p.parent.name+"/" if p.parent!=OUT else "")+p.name
        data=p.read_bytes()
        z.writestr(name,data)
        manifest["files"].append(dict(name=name,bytes=len(data),
                          sha256=hashlib.sha256(data).hexdigest()))
    z.writestr("FULL_vs_DELTA.csv",csv_bytes)
    manifest["files"].append(dict(name="FULL_vs_DELTA.csv",bytes=len(csv_bytes),
                         sha256=hashlib.sha256(csv_bytes).hexdigest()))
    z.writestr("SHA256_manifest.json",json.dumps(manifest,indent=2))
with zipfile.ZipFile(ZIP) as z:
    disallowed=[p for p in z.namelist() if p.lower().endswith(
        (".safetensors",".bin",".pt",".pth","tokenizer.json"))]
    assert not disallowed,disallowed
    assert z.testzip() is None
print("Statistics archive:",ZIP,"bytes:",ZIP.stat().st_size)
print("FULL HF:",summary["hf_full"])
print("DELTA HF:",summary["hf_delta"])
try:
    from IPython.display import FileLink,display
    display(FileLink(str(ZIP)))
except ImportError:
    pass
try:
    from google.colab import files as colab_files
    colab_files.download(str(ZIP))
    print("Browser download triggered")
except Exception as err:
    print("Browser download did not start; ZIP remains on /content:",repr(err))
